# Matching CNT's Gaussian primitives (`tab:app-primitives`, supplement)

One-to-one Hungarian matching of CNT's Gaussian primitives by position / appearance / mixed cost, fixed vs moving layout (E1); soft/dense/hard primitive-OT variants (E4); a scramble diagnostic (E3, `GEScrambleDReal`).

**Needs:** DTD 120-pair subset, the CNT checkpoint.

**Maintained runnable path:** `experiments/primitives_120.py` reproduces this; `score_only.py` rescores it (`tab:app-primitives-120`) from cached features. Kept for provenance.

# Which correspondence should texture interpolation preserve? — implementation of the 9 Sep 2026 protocol (no human study)

Runs on the real CNT pipeline (same setup cell as `finish_day` / `mechanism_study`). Everything is resumable: images go to `correspondence/imgs/…`, rows to CSVs, features to `correspondence/feats/*.npz`. Re-running a cell only computes what is missing.

| Cell | Protocol § | What it does | Decodes | A100 est. |
|---|---|---|---|---|
| 0 | — | canonical setup (verbatim) + pinned CNT commit `0726930` | | ~5 min |
| 1 | §2, §3, §10 | Gaussian-element API: elements, standardised costs, deterministic rectangular Hungarian, **fixed** / **moving** decoders, faithful-CNT row, audits of the checkout | | <1 min |
| 2 | §3, §8 | pair sets (20 CAL / 24 SENS / 40 E2 bases / 80 REPL), regime labels, endpoint-reuse clusters, cost-scale calibration, **protocol.json freeze**, E2a level calibration | ~60 | ~2 min |
| 3 | §10 | timing batch on 10 CAL pairs → extrapolated schedule | 90 | ~1 min |
| 4 | §10 checks | validation on CAL: t=0 reproduction, bijectivity, tie test, direction, determinism, fixed-tensor invariance, faithful≟P | ~200 | ~3 min |
| 5 | §3 E1 | **main matrix** on the 120 pairs: {P, A, PA} × {morph, fixA, fixB} + faithful row, t=0.5, scored | 1200 | ~20 min |
| 6 | §3 E1 | sensitivity subset (24): λ-sweep (7 λ × 3 layouts) and t-path (5 t × 4 costs × 3 layouts) | ~1500 | ~25 min |
| 7 | §4 E2a | latent counterfactuals: 40 bases × 2 appearance levels × 2 ambiguity levels × 4 moved fractions × 3 costs × 2 layouts + oracles | ~5800 | ~60 min |
| 8 | §4 E2b | real-image replication: fixed hue/contrast edit × {identity, crop-shift, 2×2 tiles, 4×4 tiles}; oracle registration control | ~2100 | ~25 min |
| 9 | §5 E3 | corrupt/restore the matching cue only (position labels; appearance descriptors as negative control), relabel check | ~1800 | ~20 min |
| 10 | §6 E4 | Gaussian-level entropic appearance rows (ε grid, variance-matched to dense OT) + all existing dense rows, scored on the 120 | ~800 | ~15 min |
| 11 | §7, §8 | analysis: paired contrasts, Holm, cost×layout and reliability interactions, cluster bootstrap, KID paired subsampling, retrieval → `summary.json` | | ~2 min |
| 12 | §11 | figures: factorial panel, reliability panel, Gaussian-vs-dense panel, λ curves, uncurated first-ID grid | | ~1 min |

**Deviations from the protocol document, recorded here and in `protocol.json`:**
1. **No human study.** For E2 the primary outcome is decoded oracle error (unchanged). For natural pairs the co-primary automatic outcomes are paired ΔKID (canonical Inception, 128 px, vs the full DTD reference, paired m-of-n subsampling) and per-pair DINO realism; balance |place−½|, endpoint retrieval with controls, LPIPS to endpoint reconstructions are secondary. "Coherence" is not claimed from automatic metrics.
2. **Appearance vs geometry split follows what the decoder consumes.** `layout_splat.py` concatenates `rot_feature` into `blob_features`, so the decoder sees it as appearance; geometry = {xy, covs, weights, compositing order}. Flag `ROT_IS_APPEARANCE` records the choice.
3. **Faithful CNT row** = reconstruction from the authors' seam-blending code (`style_transfer.py::_resolve_patch_seams`: Hungarian on plain Euclidean centre distance, linear xy/covs/features with unit renormalisation, weights 1) + paper text. No B.5 interpolation entry point exists at commit `0726930`. It differs from `P/morph` only by the metric (L2 vs squared L2) and the unit conventions; the notebook counts how often the two assignments differ.
4. **Unequal active counts** (rectangular assignment): fixed layout → every source element gets a target (unmatched → argmin of the same cost, many-to-one), unmatched target elements unused; moving layout → source order kept, unmatched source elements fade out (weight 1−t), unmatched target elements appended and fade in (weight t). Counts are logged per pair.
5. **Directions.** With symmetric costs the moving branch is the same assignment run backwards (t↔1−t) up to compositing order, so it is run A→B only and the B→A LPIPS is reported as a diagnostic; the fixed branch is run on both layouts (fixA, fixB) and the two are averaged within pair.
6. **E2a perturbation.** A purely global additive edit leaves the appearance-optimal assignment unchanged (the cross term sums to a constant over any bijection), so ambiguity is injected per element: f′ = blocknorm(f + a·u + σ·d_NN·ε_i) with u a fixed unit direction per base and ε_i per-element noise; σ is in units of the base's median nearest-neighbour feature distance. a is calibrated on CAL bases to reach fixed fractions of the natural-pair endpoint distance.
7. **E2b geometry** uses crop-shift (32 px, common support 224²) and seeded tile derangements (2×2, 4×4) rather than warps; images are re-encoded, no Gaussian ground truth is claimed.
8. **H5 (sharp vs diffuse) is out of scope** here; it belongs to the smoothing study.


In [ ]:
# ============================================================
# 0. SETUP — canonical protocol (finish_day cell 0 / mechanism cell 0, verbatim) + CNT commit pinned to the audited checkout
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile,hashlib,itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
assert torch.cuda.is_available(),'switch runtime to GPU'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'
MECH=f'{BASE}/mechanism'; os.makedirs(MECH,exist_ok=True); FIGS=f'{MECH}/figs'; os.makedirs(FIGS,exist_ok=True)
TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
t0=time.time()

# --- DTD ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd'); os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd'); src='/content/_dtd/images' if 'images' in inner else f'/content/_dtd/{inner[0]}'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
CNT_COMMIT=os.popen(f'cd {CNT} && git rev-parse HEAD').read().strip()
print(f'model ready {time.time()-t0:.0f}s | FMS={FMS} | CNT commit {CNT_COMMIT[:10]}')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def _pyr(si):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(FMS): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return sp
def build_style_maps(bf,si,fms):
    sp=_pyr(si); return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def decode_stagewise(blob_ref,sm_by_stage):
    """same decoder, but stage i receives sm_by_stage[i] (dict size->map). For E5 per-stage swaps."""
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm_by_stage[i][fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def sm_from_field(Z,L,H,W,Cc):
    it=Z.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
def _dec(d,Z,L,H,W,Cc,layout='a'): return decode_with_style_maps(d["blob_"+layout],sm_from_field(Z.float(),L,H,W,Cc))
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

# --- canonical Sinkhorn (as used in the paper) + converged log-domain Sinkhorn with duals ---
def sinkhorn_plan(x,y,eps,iters=ITERS):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def sinkhorn_log(x,y,eps,iters=3000,tol=1e-6):
    """balanced, uniform marginals, float64 log-domain; returns P, duals f,g, cost, marginal residual, iterations."""
    X=x.double(); Y=y.double(); Cm=torch.cdist(X,Y).pow(2); n,m=Cm.shape
    la=-math.log(n); lb=-math.log(m); f=torch.zeros(n,device=X.device,dtype=torch.float64); g=torch.zeros(m,device=X.device,dtype=torch.float64)
    for it in range(1,iters+1):
        f=eps*(la-torch.logsumexp((g[None,:]-Cm)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-Cm)/eps,0))
        if it%10==0:
            logP=(f[:,None]+g[None,:]-Cm)/eps; res=float((torch.exp(logP).sum(1)-1/n).abs().max()*n)
            if res<tol: break
    logP=(f[:,None]+g[None,:]-Cm)/eps; P=torch.exp(logP)
    res=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/m).abs().max()*m))
    return P,f,g,Cm,res,it
def plan_stats(P):
    P=P.double(); n=P.shape[0]; q=P/(P.sum(1,keepdim=True)+1e-300)
    Hrow=float(-(q*torch.log(q+1e-300)).sum(1).mean()); Htot=float(-(P*torch.log(P+1e-300)).sum())
    return dict(H_row=Hrow,H_total=Htot,MI_nats=math.log(n)-Hrow)
def natural_extension(x_all,y,g,eps,chunk=1024):
    """T(x)=sum_j q_j(x) y_j, q_j ∝ exp((g_j-||x-y_j||²)/eps) — dual-potential extension of a converged plan."""
    out=[]
    for i in range(0,len(x_all),chunk):
        Cm=torch.cdist(x_all[i:i+chunk].double(),y.double()).pow(2); q=torch.softmax((g[None,:]-Cm)/eps,1); out.append(q@y.double())
    return torch.cat(out)

# --- matching rules -> sampled targets (M,d); full field via canonical nn extension ---
def _targets(sf,tf,rule,seed,eps=EPS,M=NSUB,n_exact=1024,plan_kind='canonical'):
    seed=int(seed)   # numpy int64 from pandas breaks torch manual_seed
    N=sf.shape[0]; i_s=_sub(N,seed,'s')[:M]; i_t=_sub(N,seed,'t')[:M]; src,tgt=sf[i_s],tf[i_t]; info={}
    if rule in('ot','ot_sharp','ot_soft'):
        e={'ot':eps,'ot_sharp':0.005,'ot_soft':0.5}[rule] if rule!='ot' else eps
        if plan_kind=='canonical': P,_=sinkhorn_plan(src,tgt,e); info=dict(eps=e,**plan_stats(P))
        else: P,f,g,_,res,it=sinkhorn_log(src,tgt,e); info=dict(eps=e,res=res,iters=it,**plan_stats(P)); info['g']=g
        tt=((P/(P.sum(1,keepdim=True)+1e-30))@tgt.double()).float() if P.dtype==torch.float64 else (P/(P.sum(1,keepdim=True)+1e-30))@tgt
    elif rule=='exact':
        g_=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g_,device=device)[:n_exact]; is_=torch.randperm(M,generator=g_,device=device)[:n_exact]
        r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
        tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]; info=dict(n_exact=n_exact)
    elif rule=='nn': tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        X,Y=src.double(),tgt.double(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx; Yc=Y-my; Cc=X.shape[1]; I=torch.eye(Cc,device=device,dtype=torch.float64); ridge=1e-4
        Sa=(Xc.T@Xc)/(len(X)-1)+ridge*I; Sb=(Yc.T@Yc)/(len(Y)-1)+ridge*I
        def ps(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.sqrt())@V.T
        def pis(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.rsqrt())@V.T
        Sh=ps(Sa,ridge); Sih=pis(Sa,ridge); Amap=Sih@ps(Sh@Sb@Sh,0.0)@Sih; Amap=.5*(Amap+Amap.T)
        tt=(my+Xc@Amap.T).float(); info=dict(A=Amap,mu=(mx,my))
    elif rule=='uniform': tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g_=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g_,device=device)]
    elif rule=='meanshift': tt=src+(tgt.mean(0)-src.mean(0))      # CNT-style native mean shift, adapted to the interpolation contract
    else: raise ValueError(rule)
    return i_s,tt,info
def field(d,rule,seed,**kw):
    seed=int(seed)
    """full-grid target field U (N,d) with the canonical nn extension; also returns X,Y grids and plan info."""
    L,Cc,H,W,sf,tf=_grids(d); i_s,tt,info=_targets(sf,tf,rule,seed,**kw)
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return dict(U=tt[nn],X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,i_s=i_s,tt=tt,nn=nn,info=info)
def zt(fd,t): return (1-t)*fd['X']+t*fd['U']
def decode_field(d,fd,Z): return _dec(d,Z,fd['L'],fd['H'],fd['W'],fd['Cc'])

# --- fixed-layout subspace of A: X = W_A F_A ---
def splat_W(d,side='a'):
    sp=_pyr(d['si_'+side]); Wm=sp[max(FMS)][0]; K=Wm.shape[0]
    return Wm.reshape(K,-1).T.contiguous().double(), d['bf_'+side][0].double()      # (N,K),(K,d)
def projector(Wm,tol=1e-6):
    Uw,S,_=torch.linalg.svd(Wm,full_matrices=False); keep=S>S[0]*tol; return Uw[:,keep], int(keep.sum())
def proj(Q,U): U=U.double(); return Q@(Q.T@U)
def e_sub(Q,U):
    U=U.double(); Uc=U-U.mean(0,keepdim=True); off=U-proj(Q,U)
    return dict(e_sub=float(off.norm()/(Uc.norm()+1e-12)),off_rms=float(off.norm()/math.sqrt(U.numel())),cen_rms=float(Uc.norm()/math.sqrt(U.numel())))
def e_aff(X,U,seed=0):
    """affine fit U≈XA+b on half the rows, residual on held-out rows."""
    X=X.double(); U=U.double(); N=len(X); g=torch.Generator(device=X.device).manual_seed(seed); p=torch.randperm(N,generator=g,device=X.device); tr,te=p[:N//2],p[N//2:]
    Xa=torch.cat([X,torch.ones(N,1,device=X.device,dtype=torch.float64)],1)
    G=Xa[tr].T@Xa[tr]; sol=torch.linalg.solve(G+1e-6*torch.trace(G)/G.shape[0]*torch.eye(G.shape[0],device=G.device,dtype=G.dtype),Xa[tr].T@U[tr]); R=U[te]-Xa[te]@sol; Uc=U[te]-U[te].mean(0,keepdim=True)
    return dict(e_aff=float(R.norm()/(Uc.norm()+1e-12)),aff_rms=float(R.norm()/math.sqrt(R.numel())))
def commutation_err(Q,Wm,Fm,Tfun):
    """J = T(W F) - W T(F), for a frozen map T applied to blob vectors (out-of-support warning if far from cloud)."""
    X=(Wm@Fm); TX=Tfun(X.float()).double(); TF=Tfun(Fm.float()).double(); J=TX-Wm@TF
    return dict(comm=float(J.norm()/(TX-TX.mean(0)).norm()),comm_rms=float(J.norm()/math.sqrt(J.numel())))
def neighbor_disc(U,H,W):
    G=U.reshape(H,W,-1); return float(((G[:,1:]-G[:,:-1]).norm(dim=-1).mean()+(G[1:]-G[:-1]).norm(dim=-1).mean())/2)
def var_ret(U,Y): return float(U.double().var(0).sum()/Y.double().var(0).sum())
def interaction_scale(X,Y):
    """RMS of -2 Xc Yc^T (centered interaction) vs raw median cost: the calibration the plan asks for."""
    Xc=X.double()-X.double().mean(0); Yc=Y.double()-Y.double().mean(0); M_=-2*Xc@Yc.T
    return dict(inter_rms=float(M_.pow(2).mean().sqrt()),cost_med=float(torch.cdist(X.double(),Y.double()).pow(2).median()))

# --- observers (canonical) ---
import timm, lpips as lpips_lib
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
SCORE_RES=128
def at128(pil): return pil.convert('RGB').resize((SCORE_RES,SCORE_RES))
ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def kid_point(X,Y,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(len(X),len(Y)); x=X[:m]; y=Y[rng.choice(len(Y),m,replace=False)]
    kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
KS120=[int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k]; KSET=set(KS120)
DEV=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET][:24]          # development set: pair-disjoint from the 120 confirmation pairs
print('dev pairs:',DEV); print('confirmation pairs:',len(KS120))

# --- per-pair scoring bundle for a decoded midpoint (human-study proxies) ---
_anc={}
def anchors(k):
    if k in _anc: return _anc[k]
    A,B=pair_imgs(k); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B)); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]
def reencode_field(pil):
    e=cnt_load(pil); ba,bfa,sia=get_gen_inputs(e); sm=build_style_maps(bfa,sia,FMS); L=max(FMS); A=sm[L][0]; Cc=A.shape[0]
    return A.permute(1,2,0).reshape(-1,Cc)
def score_image(k,pil,Z=None,t=None):
    a=anchors(k); im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
           dA=float(np.linalg.norm(fd-a['fA'])/a['dAB']),dB=float(np.linalg.norm(fd-a['fB'])/a['dAB']))
    r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    if Z is not None:
        Xr=reencode_field(pil); r['roundtrip']=float((Xr.double()-Z.double()).norm()/(Z.double().norm()+1e-12))
    return r,fi,fd
_cache_d={}
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        A,B=pair_imgs(k); _cache_d[k]=build_d(cnt_load(A),cnt_load(B))
    return _cache_d[k]
def save_rows(rows,path): pd.DataFrame(rows).to_csv(path,index=False)
def load_rows(path,keys):
    if os.path.exists(path):
        df=pd.read_csv(path); return df.to_dict('records'), set(tuple(r[c] for c in keys) for r in df.to_dict('records'))
    return [],set()
print(f'setup done in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# 1. GAUSSIAN-ELEMENT API — elements, standardised costs, deterministic Hungarian, fixed / moving decoders, faithful-CNT row, audits
#    (protocol §2 provenance, §3 cost construction + frozen-layout definition, §10 implementation checks)
# ============================================================
from PIL import ImageEnhance
CORR=f'{BASE}/correspondence'; IMGS=f'{CORR}/imgs'; CFIG=f'{CORR}/figs'; FEATS=f'{CORR}/feats'
for p in [CORR,IMGS,CFIG,FEATS]: os.makedirs(p,exist_ok=True)
CNT_PIN='07269302610c92ce38e2dbbe31c492fd09e1c73e'; assert CNT_COMMIT==CNT_PIN,f'CNT checkout {CNT_COMMIT[:10]} != pinned {CNT_PIN[:10]}'
CKPT_SHA=hashlib.sha256(open(f'{CNT}/ckpts/finetuned-model256.ckpt','rb').read()).hexdigest()[:16]
ACTIVE_THR=0.5             # layout_net.py inference branch: weights = round(exist_prob) -> admissible element := weight > 0.5
ROT_IS_APPEARANCE=True     # layout_splat.py (texton_encoder=disentangled_local_texton) concatenates rot_feature into blob_features -> consumed as appearance
RUN=dict(TIMING=True,VALIDATE=True,E1=True,SENS=True,E2a=True,E2b=True,E3=True,E4=True,REPL=False)   # REPL: 80-pair replication set (§3)

def tl(blob): return blob['layer_tree_levels'][-1][0]
def elements(blob):
    """active Gaussian elements in native slot order. pos = normalised image coords in [0,1] (splat: (xy+3)/6 * img_size)."""
    T=tl(blob); w=T['weights'][0,:,0]; idx=torch.nonzero(w>ACTIVE_THR)[:,0]
    E=dict(idx=idx,xy=T['xy'][0,idx],covs=T['covs'][0,idx],w=w[idx],ptf=T['per_texel_feature'][0,idx],rot=T['rot_feature'][0,idx],bg=blob['bg_features'][0],K=int(w.shape[0]),n=int(len(idx)))
    E['f']=torch.cat([E['ptf'],E['rot']],-1) if ROT_IS_APPEARANCE else E['ptf']; E['pos']=(E['xy']+3)/6; return E
def pair_elems(k):
    d=get_d(k); return d,elements(d['blob_a']),elements(d['blob_b'])

# --- encode ONCE, cache on Drive (protocol §3: never re-encode); bf/si come from the deterministic splat of the cached blob ---
ENC=f'{CORR}/enc'; os.makedirs(ENC,exist_ok=True); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
def _to(x,dev): return {k_:_to(v,dev) for k_,v in x.items()} if isinstance(x,dict) else ([_to(v,dev) for v in x] if isinstance(x,list) else (x.to(dev) if torch.is_tensor(x) else x))
def _d_from_blobs(ba,bb):
    with torch.no_grad():
        La=model.splat(ba)['layer_level_layouts'][-1][0]; Lb=model.splat(bb)['layer_level_layouts'][-1][0]
    return dict(blob_a=ba,bf_a=La['blob_features'].clone(),si_a=La['score_img'].clone(),blob_b=bb,bf_b=Lb['blob_features'].clone(),si_b=Lb['score_img'].clone())
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        fp=f'{ENC}/{k:04d}.pt'
        if os.path.exists(fp): E_=torch.load(fp,map_location='cpu',weights_only=False); ba,bb=_to(E_['blob_a'],device),_to(E_['blob_b'],device)
        else:
            A,B=pair_imgs(k)
            with torch.no_grad(): ba=model.encode(cnt_load(A)); bb=model.encode(cnt_load(B))
            torch.save(dict(blob_a=_to(ba,'cpu'),blob_b=_to(bb,'cpu')),fp)
        _cache_d[k]=_d_from_blobs(ba,bb)
    return _cache_d[k]
_cache_d.clear(); _anc.clear()

# --- audit of the checkout on one pair: dims, norms (decides whether renormalisation is a no-op at the endpoints), weight values, xy range ---
_d0,_E0,_=pair_elems(KS120[0]); _np_=_E0['ptf'].norm(dim=-1); _nr_=_E0['rot'].norm(dim=-1)
RENORM_PTF=bool(((_np_-1).abs()<1e-3).all()); RENORM_ROT=bool(((_nr_-1).abs()<1e-3).all())
DP=int(_E0['ptf'].shape[-1]); DR=int(_E0['rot'].shape[-1]); assert _d0['bf_a'].shape[-1]==DP+DR and _E0['bg'].shape[-1]==DP+DR
_w_=torch.unique(tl(_d0['blob_a'])['weights']).tolist()
print(f"slots K={_E0['K']} active={_E0['n']} | per_texel {DP}-d norm {_np_.mean():.4f}±{_np_.std():.4f} (renorm={RENORM_PTF}) | rot {DR}-d norm {_nr_.mean():.4f} (renorm={RENORM_ROT}) | bg norm {float(_E0['bg'].norm()):.3f}")
print(f"weights take values {_w_[:6]} | xy range [{float(tl(_d0['blob_a'])['xy'].min()):.2f},{float(tl(_d0['blob_a'])['xy'].max()):.2f}] | ckpt sha {CKPT_SHA}")
assert set(round(v,3) for v in _w_)<={0.0,1.0},'inference weights are not binary: revisit ACTIVE_THR and the fade policy'
def _nptf(x): return F.normalize(x,p=2,dim=-1) if RENORM_PTF else x
def _nrot(x): return F.normalize(x,p=2,dim=-1) if RENORM_ROT else x
def _lerp(a,b,t): return (1-t)*a+t*b

# --- costs (§3): squared distances; scales frozen in cell 2 ---
SC={'s_pos':None,'s_app':None}
def cost_mats(ES,ET):
    return torch.cdist(ES['pos'].double(),ET['pos'].double()).pow(2), torch.cdist(ES['f'].double(),ET['f'].double()).pow(2)
def mixed_cost(Cp,Ca,lam):
    assert SC['s_app'] and SC['s_pos'],'cost scales not frozen yet (run cell 2)'
    return (1-lam)*Ca/SC['s_app']+lam*Cp/SC['s_pos']
def solve(C):
    """deterministic rectangular Hungarian (scipy). r,c sorted by source row. unA/unB = unmatched elements (unequal active counts).
       map_full: a target for EVERY source element - matched -> partner, unmatched -> argmin of the same cost (many-to-one; fixed-layout policy)."""
    Cn=C.detach().cpu().numpy(); r,c=linear_sum_assignment(Cn); o=np.argsort(r); r,c=r[o],c[o]
    nA,nB=Cn.shape; mA=np.zeros(nA,bool); mA[r]=True; mB=np.zeros(nB,bool); mB[c]=True
    mf=np.empty(nA,np.int64); mf[r]=c; unA=np.nonzero(~mA)[0]; unB=np.nonzero(~mB)[0]
    if len(unA): mf[unA]=Cn[unA].argmin(1)
    T=lambda a:torch.as_tensor(np.asarray(a),device=device,dtype=torch.long)
    return dict(r=T(r),c=T(c),unA=T(unA),unB=T(unB),map_full=T(mf),matched=T(mA.astype(np.int64)).bool(),cost=float(Cn[r,c].sum()),nA=int(nA),nB=int(nB))
def identity_asg(n):
    ar=torch.arange(n,device=device); return dict(r=ar,c=ar,unA=ar[:0],unB=ar[:0],map_full=ar,matched=torch.ones(n,dtype=torch.bool,device=device),cost=0.0,nA=n,nB=n)
LAM={'A':0.0,'PA':0.5,'P':1.0}
def solve_cost(ES,ET,name,lam=None,Cp=None,Ca=None):
    """name in {'P','A','PA','F'} or explicit lam. 'F' = faithful CNT: Hungarian on plain Euclidean centre distance in raw xy units
       (style_transfer.py::_resolve_patch_seams). Cp/Ca may be passed pre-corrupted (E3)."""
    if Cp is None or Ca is None: Cp,Ca=cost_mats(ES,ET)
    C=torch.cdist(ES['xy'].double(),ET['xy'].double()) if name=='F' else mixed_cost(Cp,Ca,LAM[name] if lam is None else lam)
    return solve(C),Cp,Ca
def asg_metrics(asg,Cp,Ca,ref=None):
    r,c=asg['r'],asg['c']; m=dict(n_matched=int(len(r)),nA=asg['nA'],nB=asg['nB'],A_cost=float(Ca[r,c].mean()/SC['s_app']),G_cost=float(Cp[r,c].mean()/SC['s_pos']))
    if ref is not None: m['overlap_ref']=float((asg['map_full']==ref['map_full']).float().mean())
    return m

# --- decoders ---
def fixed_bf(d,side,ES,fT,bgT,t):
    """FROZEN LAYOUT feature rows: side's complete geometry stays (all K slots incl. inactive, native order, native score_img); only the appearance
       rows of the active elements change: f_i(t)=(1-t) f_i^S + t fT_i, blocks renormalised as in the native code; bg row lerped. fT is (n,DP+DR)."""
    bf=d['bf_'+side].clone(); fS=torch.cat([ES['ptf'],ES['rot']],-1); f=_lerp(fS,fT,t)
    rot=_nrot(f[:,DP:]) if ROT_IS_APPEARANCE else fS[:,DP:]
    bf[0,1+ES['idx']]=torch.cat([_nptf(f[:,:DP]),rot],-1); bf[0,0]=_lerp(ES['bg'],bgT,t); return bf
def decode_fixed_feats(d,side,ES,fT,bgT,t): return decode_with_style_maps(d['blob_'+side],build_style_maps(fixed_bf(d,side,ES,fT,bgT,t),d['si_'+side],FMS))
def decode_fixed(d,side,ES,ET,asg,t):
    return decode_fixed_feats(d,side,ES,torch.cat([ET['ptf'],ET['rot']],-1)[asg['map_full']],ET['bg'],t)
def moving_params(ES,ET,asg,t):
    """MOVING LAYOUT (native seam-code rule): linear xy / covs / features (+renorm), weights 1. Source elements keep native slot order
       (t=0 reproduces the source); unmatched source elements fade out (1-t), unmatched target elements are appended in raster order and fade in (t)."""
    m=asg['matched'][:,None]; mf=asg['map_full']
    xy=torch.where(m,_lerp(ES['xy'],ET['xy'][mf],t),ES['xy']); covs=torch.where(m[:,:,None],_lerp(ES['covs'],ET['covs'][mf],t),ES['covs'])
    ptf=torch.where(m,_nptf(_lerp(ES['ptf'],ET['ptf'][mf],t)),ES['ptf']); rot=torch.where(m,_nrot(_lerp(ES['rot'],ET['rot'][mf],t)),ES['rot'])
    w=torch.where(m[:,0],torch.ones_like(ES['w']),(1-t)*torch.ones_like(ES['w'])); u=asg['unB']
    if len(u)>1:                                                        # fade-in extras composited in raster order (y, then x): label-free, deterministic
        o=torch.argsort(ET['xy'][u,0],stable=True); o=o[torch.argsort(ET['xy'][u,1][o],stable=True)]; u=u[o]
    xy=torch.cat([xy,ET['xy'][u]]); covs=torch.cat([covs,ET['covs'][u]]); ptf=torch.cat([ptf,ET['ptf'][u]]); rot=torch.cat([rot,ET['rot'][u]])
    w=torch.cat([w,float(t)*torch.ones(len(u),device=device,dtype=w.dtype)])
    return xy,covs,w,ptf,rot,_lerp(ES['bg'],ET['bg'],t)
def make_blob(base,xy,covs,w,ptf,rot,bg):
    T=dict(tl(base)); T.update(xy=xy[None],covs=covs[None],weights=w[None,:,None],exist_prob=w[None,:,None],per_texel_feature=ptf[None],rot_feature=rot[None])
    b=dict(base); ltl=base['layer_tree_levels']; ltl2=dict(ltl) if isinstance(ltl,dict) else list(ltl); ltl2[-1]=[T]
    b['layer_tree_levels']=ltl2; b['bg_features']=bg[None]; return b
def decode_blob(blob):
    with torch.no_grad():
        lay=model.splat(blob)['layer_level_layouts'][-1][0]; return decode_with_style_maps(blob,build_style_maps(lay['blob_features'],lay['score_img'],FMS))
def decode_moving(d,ES,ET,asg,t,side='a'): return decode_blob(make_blob(d['blob_'+side],*moving_params(ES,ET,asg,t)))

def anchors(k):
    """endpoint reconstructions decoded from the SAME cached blobs every cell uses (make_recon would re-encode)"""
    if k in _anc: return _anc[k]
    d=get_d(k); rA=to_pil(decode_blob(d['blob_a'])); rB=to_pil(decode_blob(d['blob_b'])); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]

# --- E2a construction (§4): same elements, per-element appearance edit, nested seeded derangement of complete supports ---
def e2a_target(E,a,sig,mfrac,seed):
    n=E['n']; Dm=E['f'].shape[-1]; g=torch.Generator(device=device).manual_seed(int(seed))
    u=torch.randn(Dm,generator=g,device=device); u=u/u.norm(); eps=torch.randn(n,Dm,generator=g,device=device)/math.sqrt(Dm)
    Dn=torch.cdist(E['f'].double(),E['f'].double()); Dn.fill_diagonal_(float('inf')); dnn=float(Dn.min(1).values.median())
    f=E['f']+a*u[None]+sig*dnn*eps
    if ROT_IS_APPEARANCE: ptf,rot=_nptf(f[:,:DP]),_nrot(f[:,DP:])
    else: ptf,rot=_nptf(f),E['rot']
    order=torch.randperm(n,generator=g,device=device); ar=torch.arange(n,device=device); perm=ar.clone()
    nm=max(2,int(math.ceil(mfrac*n))) if mfrac>0 else 0
    if nm>=2: S=order[:nm]; perm[S]=S.roll(-1)                     # element S[j] takes the complete support (xy,covs) of S[j+1]
    moved=perm!=ar; xy=E['xy'][perm]; covs=E['covs'][perm]
    disp=float((E['pos'][perm]-E['pos']).norm(dim=-1)[moved].mean()) if nm>=2 else 0.0
    return dict(idx=ar,xy=xy,covs=covs,w=torch.ones(n,device=device),ptf=ptf,rot=rot,bg=E['bg'],f=torch.cat([ptf,rot],-1) if ROT_IS_APPEARANCE else ptf,
                pos=(xy+3)/6,n=n,K=n,dnn=dnn,n_moved=int(moved.sum()),disp=disp,u=u)
def blob_of(d,E): return make_blob(d['blob_a'],E['xy'],E['covs'],E['w'],E['ptf'],E['rot'],E['bg'])

# --- scoring / persistence helpers ---
def score_generic(pil,fA=None,fB=None):
    im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]))
    if fA is not None:
        dAB=float(np.linalg.norm(fA-fB))+1e-9; r['dA']=float(np.linalg.norm(fd-fA)/dAB); r['dB']=float(np.linalg.norm(fd-fB)/dAB); r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    return r,fi,fd
def dino1(pil): return dino_feats([at128(pil)])[0]
def maxdiff(a,b): return float((a-b).abs().max())
def feat_load(name):
    p=f'{FEATS}/{name}.npz'; return {k:v for k,v in np.load(p).items()} if os.path.exists(p) else {}
def feat_save(name,D): np.savez(f'{FEATS}/{name}.npz',**D)
def img_path(group,cell,k,t=None):
    p=f'{IMGS}/{group}/{cell}' if t is None else f'{IMGS}/{group}/{cell}_t{t}'; os.makedirs(p,exist_ok=True); return f'{p}/{k:04d}.png'
def sync(): torch.cuda.synchronize()
CSV1=f'{CORR}/E1_main.csv'; AMC=f'{CORR}/assignment_metrics.csv'; CSV1S=f'{CORR}/E1_sens.csv'; CSV2=f'{CORR}/E2a.csv'; ECSV=f'{CORR}/E2a_endpoints.csv'; CSV2B=f'{CORR}/E2b.csv'; CSV3=f'{CORR}/E3.csv'; CSV4=f'{CORR}/E4_gauss_sinkhorn.csv'
print('API ready')

In [ ]:
# ============================================================
# 2. PAIR SETS, REGIME LABELS, ENDPOINT-REUSE CLUSTERS, COST-SCALE CALIBRATION, PROTOCOL FREEZE (§3, §8) + E2a level calibration (§4)
# ============================================================
SEED=20260909
POOL=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET]; assert POOL[:24]==DEV,'dev-set drift vs mechanism study'
CAL=POOL[24:44]                                                                     # 20 calibration pairs: scales + implementation validation (disjoint from 120 and DEV)
_r=np.random.RandomState(SEED); SENS=sorted(int(x) for x in _r.choice(KS120,24,replace=False))   # 24-pair sensitivity subset of the 120, by seed, before any scoring
_rest=POOL[44:]; _perm=_r.permutation(len(_rest))
E2_BASES=sorted(int(_rest[i]) for i in _perm[:40])                                  # 40 independent base textures (A image of these manifest pairs)
REPL=sorted(int(_rest[i]) for i in _perm[40:120])                                   # 80-pair disjoint replication set (only if RUN['REPL'])
MAIN_KS=KS120+(REPL if RUN['REPL'] else []); SAVE_BASES=set(E2_BASES[:8])           # E2 images kept on disk only for the first 8 bases (uncurated)
PINFO={int(p_['k']):p_ for p_ in MAN_ALL}
def nameA(k): return os.path.basename(str(PINFO[k].get('fA','')))
def nameB(k): return os.path.basename(str(PINFO[k].get('fB','')))
def cat_of(f): return os.path.basename(os.path.dirname(str(f))) if f else 'unknown'
PERIODIC={'banded','chequered','crosshatched','dotted','grid','honeycombed','knitted','lined','meshed','perforated','polka-dotted','striped','waffled','woven','zigzagged','braided','interlaced','lacelike','pleated','studded','paisley','spiralled','swirly'}
def regime(c): return 'periodic' if c in PERIODIC else 'stochastic'
CAT={k:dict(A=cat_of(PINFO[k].get('fA')),B=cat_of(PINFO[k].get('fB'))) for k in PINFO}
def pair_regime(k):
    a,b=regime(CAT[k]['A']),regime(CAT[k]['B']); return 'both_periodic' if a==b=='periodic' else ('both_stochastic' if a==b=='stochastic' else 'mixed')
def _components(ks):
    """connected components of the endpoint-reuse graph (pairs sharing an image) -> bootstrap clusters (§3, §8)"""
    par={}
    def f(x):
        par.setdefault(x,x)
        while par[x]!=x: par[x]=par[par[x]]; x=par[x]
        return x
    for k in ks: par[f(nameA(k))]=f(nameB(k))
    roots=sorted({f(nameA(k)) for k in ks}); ids={r:i for i,r in enumerate(roots)}; return {k:ids[f(nameA(k))] for k in ks}
CLUSTER=_components(MAIN_KS); n_comp=len(set(CLUSTER.values()))
print(f'main {len(MAIN_KS)} pairs | endpoint-reuse components {n_comp} ({"no reuse" if n_comp==len(MAIN_KS) else "REUSE PRESENT -> cluster bootstrap"}) | regimes {pd.Series([pair_regime(k) for k in MAIN_KS]).value_counts().to_dict()}')
DTD_BY_NAME={os.path.basename(f):f for f in glob.glob(f'{LOCAL}/*/*.jpg')}

# --- cost-scale calibration on CAL (§3): global robust scales = median over pairs of the per-pair median admissible edge cost; frozen ---
PROTO=f'{CORR}/protocol.json'
if os.path.exists(PROTO): SC.update(json.load(open(PROTO))['scales'])
else:
    mp,ma,ranges=[],[],[]
    for k in CAL:
        d,EA,EB=pair_elems(k); Cp,Ca=cost_mats(EA,EB); mp.append(float(Cp.median())); ma.append(float(Ca.median()))
        ranges.append(dict(k=k,nA=EA['n'],nB=EB['n'],pos_min=float(Cp.min()),pos_med=float(Cp.median()),pos_max=float(Cp.max()),app_min=float(Ca.min()),app_med=float(Ca.median()),app_max=float(Ca.max())))
    SC['s_pos']=float(np.median(mp)); SC['s_app']=float(np.median(ma)); pd.DataFrame(ranges).to_csv(f'{CORR}/calibration_costs.csv',index=False)
    print(pd.DataFrame(ranges).describe().loc[['min','50%','max']].round(4))
assert SC['s_pos']>1e-8 and SC['s_app']>1e-8,'zero scale: calibration problem, do not divide'
print('frozen scales',SC)

# --- E2a level calibration on CAL bases (§4): appearance levels a s.t. decoded endpoint distance ≈ {0.35, 0.70} × natural-pair median dAB; σ in d_NN units ---
A_GRID=[0.1,0.2,0.35,0.5,0.7,1.0,1.4]; A_TARGETS=[0.35,0.70]; SIG_LEVELS=[0.5,1.5]; M_FRACS=[0.0,0.25,0.5,1.0]
if not os.path.exists(PROTO):
    nat=np.median([anchors(k)['dAB'] for k in CAL]); rel={a:[] for a in A_GRID}
    for k in CAL[:10]:
        d,EA,_=pair_elems(k); f0=dino1(to_pil(decode_blob(d['blob_a'])))
        for a in A_GRID:
            ET=e2a_target(EA,a,0.0,0.0,k); rel[a].append(float(np.linalg.norm(dino1(to_pil(decode_blob(blob_of(d,ET))))-f0)/nat))
    med={a:float(np.median(v)) for a,v in rel.items()}; A_LEVELS=[min(A_GRID,key=lambda a:abs(med[a]-tg)) for tg in A_TARGETS]
    print('decoded endpoint distance / natural median, per a:',{a:round(m,3) for a,m in med.items()},'-> A_LEVELS',A_LEVELS)
else: A_LEVELS=json.load(open(PROTO))['e2a']['a_levels']

LAMS_SENS=[0.0,0.1,0.25,0.5,0.75,0.9,1.0]; TS=[0.0,0.25,0.5,0.75,1.0]; E3_FRACS=[0.25,0.5,1.0]; EPS_GRID=[0.003,0.01,0.03,0.1,0.3,1.0]
proto=dict(version='2026-09-09 protocol, human study removed',frozen_at=time.strftime('%Y-%m-%d %H:%M'),cnt_commit=CNT_COMMIT,ckpt_sha256_16=CKPT_SHA,img_size=IMG_SIZE,score_res=SCORE_RES,
    flags=dict(ACTIVE_THR=ACTIVE_THR,ROT_IS_APPEARANCE=ROT_IS_APPEARANCE,RENORM_PTF=RENORM_PTF,RENORM_ROT=RENORM_ROT,cov_interp='linear (native seam code; SPD cone is convex)',bg_interp='linear, no renorm',
               unequal_count_policy=dict(fixed='unmatched source -> argmin of the same cost (many-to-one); unmatched target unused',moving='source native order; unmatched source fade 1-t; unmatched target appended, fade t'),
               direction_policy='moving A->B only (symmetric cost); fixed on both layouts, averaged within pair',solver='scipy linear_sum_assignment, rectangular, rows sorted'),
    scales=SC,seed=SEED,pairs=dict(CAL=CAL,SENS=SENS,E2_BASES=E2_BASES,REPL=REPL,MAIN=KS120),
    e1=dict(lams_main=LAM,lams_sens=LAMS_SENS,ts=TS,primary_t=0.5),e2a=dict(a_levels=A_LEVELS,a_targets=A_TARGETS,sig_levels_dnn=SIG_LEVELS,moved_fracs=M_FRACS),
    e2b=dict(hue_deg=[20,50],contrast=1.15,shift_px=32,tiles=[2,4]),e3=dict(fracs=E3_FRACS),e4=dict(eps_grid_normalised=EPS_GRID,match_rule='ε whose median retained variance is closest to dense OT(ε=0.05) median retained variance'),
    outcomes=dict(natural_coprimary=['paired ΔKID (Inception, 128px, vs full DTD ref, m-of-n paired subsampling)','per-pair DINO realism'],natural_secondary=['|place-0.5|','endpoint retrieval MRR with controls','LPIPS to endpoint reconstructions','Inception realism'],
                  e2a_primary='decoded oracle error (LPIPS to construction oracle under the same layout policy)',e2a_mechanistic='ancestry recovery'),
    hypotheses=dict(H1='A vs PA and A vs P on natural pairs, per layout',H2='Δgeo(aligned)-Δgeo(disrupted) within base/edit',H3='H2 interaction larger under ambiguous appearance',H4='cost×layout interaction'),
    multiplicity='Holm over the 4 H1 realism contrasts; all effects with 95% cluster-bootstrap CIs')
if os.path.exists(PROTO):
    old=json.load(open(PROTO))
    for key in ['scales','pairs','flags','seed','e2a','e1','e3','e4']: assert old[key]==json.loads(json.dumps(proto[key])),f'protocol drift in {key}: the frozen protocol.json wins; delete it only to re-freeze deliberately'
    proto=old; print('protocol loaded (frozen',proto['frozen_at'],')')
else: json.dump(proto,open(PROTO,'w'),indent=1); print('protocol FROZEN ->',PROTO)
pd.DataFrame([dict(k=k,set=('MAIN' if k in KSET else 'REPL' if k in REPL else 'CAL' if k in CAL else 'E2' if k in E2_BASES else 'DEV'),catA=CAT[k]['A'],catB=CAT[k]['B'],regime=pair_regime(k),cluster=CLUSTER.get(k,-1),fA=nameA(k),fB=nameB(k),sens=int(k in SENS))
              for k in KS120+REPL+CAL+E2_BASES+DEV]).to_csv(f'{CORR}/pairs.csv',index=False)
print('pairs.csv written')


In [ ]:
# ============================================================
# 3. TIMING BATCH (§10): 10 CAL pairs, encode / assignment / decode(fixed, moving) / scoring separately -> extrapolated schedule
# ============================================================
if RUN['TIMING']:
    T=dict(encode=0.,assign=0.,dec_fixed=0.,dec_moving=0.,score=0.); n=0; _cache_d.clear()
    for k in CAL[:10]:
        sync(); a=time.time(); d,EA,EB=pair_elems(k); sync(); T['encode']+=time.time()-a
        for c in ['P','A','PA']:
            a=time.time(); asg,Cp,Ca=solve_cost(EA,EB,c); T['assign']+=time.time()-a
            sync(); a=time.time(); o1=decode_fixed(d,'a',EA,EB,asg,0.5); sync(); T['dec_fixed']+=time.time()-a
            a=time.time(); o2=decode_moving(d,EA,EB,asg,0.5); sync(); T['dec_moving']+=time.time()-a
            a=time.time(); score_image(k,to_pil(o1)); score_image(k,to_pil(o2)); sync(); T['score']+=time.time()-a; n+=2
    per=dict(encode_per_pair=T['encode']/10,assign_per_solve=T['assign']/30,decode_fixed=T['dec_fixed']/30,decode_moving=T['dec_moving']/30,score_per_image=T['score']/n)
    print({k:f'{v*1000:.0f} ms' for k,v in per.items()})
    dec=(per['decode_fixed']+per['decode_moving'])/2; est=dict(E1=len(MAIN_KS)*10*(dec+per['score_per_image']),SENS=24*(21+60)*(dec+per['score_per_image']),
        E2a=40*16*(8+3)*(dec+per['score_per_image']),E2b=40*2*4*7*(dec+per['score_per_image']),E3=(24+24)*(3*2+1)*6*(dec+per['score_per_image']),E4=120*7*(dec+per['score_per_image']))
    print('extrapolated minutes:',{k:round(v/60,1) for k,v in est.items()},'| total',round(sum(est.values())/60,1)); json.dump(dict(per=per,est_min={k:v/60 for k,v in est.items()}),open(f'{CORR}/timing.json','w'),indent=1)


In [ ]:
SC.update(json.load(open(f'{CORR}/protocol.json'))['scales']); print(SC)

In [ ]:
torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False   # full fp32: tiny input differences no longer flip conv rounding

In [ ]:
# ============================================================
# 4. VALIDATION on CAL (§10 minimum implementation checks)
# ============================================================
if RUN['VALIDATE']:
    V=[]
    for k in CAL[:10]:
        d,EA,EB=pair_elems(k); rA=decode_blob(d['blob_a']); rB=decode_blob(d['blob_b']); row=dict(k=k,nA=EA['n'],nB=EB['n'])
        # (0) encoder reproducibility (a second encode of the same PNG) and the effect of feature renormalisation on the t=0 rows
        with torch.no_grad(): b2=model.encode(cnt_load(pair_imgs(k)[0]))
        T1,T2=tl(d['blob_a']),tl(b2); row['encode_repro_xy']=maxdiff(T1['xy'],T2['xy']); row['encode_repro_feat']=maxdiff(T1['per_texel_feature'],T2['per_texel_feature']); row['encode_repro_img']=maxdiff(rA,make_recon(pair_imgs(k)[0]))
        row['bf_t0_maxdiff']=maxdiff(fixed_bf(d,'a',EA,torch.cat([EA['ptf'],EA['rot']],-1),EA['bg'],0.0),d['bf_a'])
        asgs={c:solve_cost(EA,EB,c)[0] for c in ['P','A','PA','F']}
        # (1) endpoint reproduction: fixed t=0 and moving t=0 must reproduce the cached blob's recon (mean-abs and LPIPS, not max: TF32/rounding
        #     can flip isolated pixels); moving t=1 vs recon B differs by compositing order + fades
        o_f=decode_fixed(d,'a',EA,EB,asgs['A'],0.0); o_m=decode_moving(d,EA,EB,asgs['A'],0.0)
        row['fixed_t0_maxdiff']=maxdiff(o_f,rA); row['fixed_t0_meandiff']=float((o_f-rA).abs().mean()); row['fixed_t0_lpips']=lpips_d(to_pil(o_f),to_pil(rA))
        row['moving_t0_maxdiff']=maxdiff(o_m,rA); row['moving_t0_meandiff']=float((o_m-rA).abs().mean()); row['moving_t0_lpips']=lpips_d(to_pil(o_m),to_pil(rA))
        row['moving_t1_lpips_vs_reconB']=lpips_d(to_pil(decode_moving(d,EA,EB,asgs['A'],1.0)),to_pil(rB))
        # (2) bijective on the matched support, targets admissible, no inactive-slot leakage
        row['bijective']=all(len(torch.unique(a['c']))==len(a['c']) and int(a['map_full'].max())<EB['n'] for a in asgs.values())
        # (3) tie test: relabel B (columns) with a seeded permutation, solve, undo -> identical map or equal cost
        g=np.random.RandomState(k); pb=g.permutation(EB['n']); EBp={kk:(v[torch.as_tensor(pb,device=device)] if kk!='bg' and torch.is_tensor(v) and v.ndim>=1 and v.shape[0]==EB['n'] else v) for kk,v in EB.items()}
        ties=0; changed=0
        for c in ['P','A','PA']:
            a2=solve_cost(EA,EBp,c)[0]; back=torch.as_tensor(pb,device=device)[a2['map_full']]
            if not torch.equal(back,asgs[c]['map_full']): changed+=1; ties+=int(abs(a2['cost']-asgs[c]['cost'])<1e-6*max(1,abs(asgs[c]['cost'])))
        row['relabel_changed']=changed; row['relabel_equal_cost_ties']=ties
        # (4) B-relabel invariance of the decoded image (physical result must not depend on B's slot labels)
        row['relabel_img_maxdiff']=maxdiff(decode_moving(d,EA,EBp,solve_cost(EA,EBp,'A')[0],0.5),decode_moving(d,EA,EB,asgs['A'],0.5)) if changed==0 else float('nan')
        # (5) determinism
        row['determinism_maxdiff']=maxdiff(decode_moving(d,EA,EB,asgs['PA'],0.5),decode_moving(d,EA,EB,asgs['PA'],0.5))
        # (6) fixed-layout tensors: across costs and t only the appearance rows of active slots may differ; bg row and inactive rows follow the same rule
        keep=torch.ones(d['bf_a'].shape[1],dtype=torch.bool,device=device); keep[1+EA['idx']]=False; keep[0]=False
        bfs=[fixed_bf(d,'a',EA,torch.cat([EB['ptf'],EB['rot']],-1)[asgs[c]['map_full']],EB['bg'],t) for c in ['P','A','PA'] for t in [0.25,0.5]]
        row['fixed_inactive_rows_untouched']=all(torch.equal(b_[0,keep],d['bf_a'][0,keep]) for b_ in bfs)
        # (7) direction: moving A->B at 0.5 vs B->A at 0.5 (same multiset of interpolated elements, different compositing order / fade side)
        asgT=solve_cost(EB,EA,'A')[0]; row['dir_lpips_moving']=lpips_d(to_pil(decode_moving(d,EA,EB,asgs['A'],0.5)),to_pil(decode_moving(d,EB,EA,asgT,0.5,side='b')))
        # (8) faithful vs P/morph assignment agreement; (9) A vs P disagreement (is there anything to test?)
        row['F_eq_P']=bool(torch.equal(asgs['F']['map_full'],asgs['P']['map_full'])); row['A_vs_P_overlap']=float((asgs['A']['map_full']==asgs['P']['map_full']).float().mean())
        V.append(row)
    V=pd.DataFrame(V); V.to_csv(f'{CORR}/validation.csv',index=False); print(V.round(4).to_string())
    print(f"encoder reproducibility (2nd encode): xy {V.encode_repro_xy.max():.2e} feat {V.encode_repro_feat.max():.2e} image {V.encode_repro_img.max():.3f} -> {'deterministic' if V.encode_repro_img.max()<1e-4 else 'NOT bit-reproducible: this is why every cell reads the cached encodings'}")
    print(f"t=0 reproduction: max {max(V.fixed_t0_maxdiff.max(),V.moving_t0_maxdiff.max()):.2e} | mean {max(V.fixed_t0_meandiff.max(),V.moving_t0_meandiff.max()):.2e} | LPIPS {max(V.fixed_t0_lpips.max(),V.moving_t0_lpips.max()):.4f}")
    assert (V.fixed_t0_meandiff<1e-3).all() and (V.moving_t0_meandiff<1e-3).all() and (V.fixed_t0_lpips<0.01).all() and (V.moving_t0_lpips<0.01).all(),f't=0 does not reproduce the cached source blob (bf diff {V.bf_t0_maxdiff.max():.2e})'
    assert (V.relabel_img_maxdiff.fillna(0)<1e-4).all(),'B-relabel changed the decoded image'
    assert V.bijective.all() and (V.determinism_maxdiff==0).all() and V.fixed_inactive_rows_untouched.all()
    print(f"faithful==P on {V.F_eq_P.mean():.0%} of pairs | A vs P mean edge overlap {V.A_vs_P_overlap.mean():.2f} | moving t=1 LPIPS to recon B {V.moving_t1_lpips_vs_reconB.median():.3f} (compositing-order + fade effect)")

In [ ]:
# ============================================================
# 5. E1 MAIN MATRIX (§3): {P, A, PA} × {morph, fixA, fixB} + faithful row on the 120 (+REPL) pairs at t=0.5. Scored with the canonical bundle.
#    assignment_metrics.csv: per (k, dir, cost) normalised costs, counts, edge overlap with the A-only and P-only assignments.
# ============================================================
CELLS1=[(c,l) for c in ['P','A','PA','F'] for l in ['morph','fixA','fixB'] if not (c=='F' and l!='morph')]
CSV1=f'{CORR}/E1_main.csv'; rows,have=load_rows(CSV1,['k','cell']); AMC=f'{CORR}/assignment_metrics.csv'; arows,ahave=load_rows(AMC,['k','dir','cost'])
FE1=feat_load('E1'); t0=time.time()
if RUN['E1']:
    for n_,k in enumerate(MAIN_KS):
        if all((k,f'{c}_{l}') in have for c,l in CELLS1): continue
        d,EA,EB=pair_elems(k); anc=anchors(k); A={}
        for dr,(ES,ET) in {'a2b':(EA,EB),'b2a':(EB,EA)}.items():
            A[dr]={c:solve_cost(ES,ET,c) for c in ['P','A','PA','F']}
            for c,(asg,Cp,Ca) in A[dr].items():
                if (k,dr,c) not in ahave:
                    m=asg_metrics(asg,Cp,Ca); m['overlap_A']=float((asg['map_full']==A[dr]['A'][0]['map_full']).float().mean()); m['overlap_P']=float((asg['map_full']==A[dr]['P'][0]['map_full']).float().mean())
                    arows.append(dict(k=k,dir=dr,cost=c,regime=pair_regime(k),**m))
        for c,l in CELLS1:
            cell=f'{c}_{l}'; fp=img_path('E1',cell,k)
            if not os.path.exists(fp):
                if l=='morph': out=decode_moving(d,EA,EB,A['a2b'][c][0],0.5)
                elif l=='fixA': out=decode_fixed(d,'a',EA,EB,A['a2b'][c][0],0.5)
                else: out=decode_fixed(d,'b',EB,EA,A['b2a'][c][0],0.5)
                to_pil(out).save(fp)
            if (k,cell) not in have:
                pil=Image.open(fp).convert('RGB'); s,fi,fd=score_image(k,pil); FE1[f'{cell}|{k}|inc']=fi; FE1[f'{cell}|{k}|dino']=fd
                rows.append(dict(k=k,cell=cell,cost=c,layout=l,t=0.5,regime=pair_regime(k),cluster=CLUSTER[k],**s,lpips_rA=lpips_d(pil,anc['rA']),lpips_rB=lpips_d(pil,anc['rB'])))
        if n_%10==0: save_rows(rows,CSV1); save_rows(arows,AMC); feat_save('E1',FE1); print(f'{n_+1}/{len(MAIN_KS)} {time.time()-t0:.0f}s')
    save_rows(rows,CSV1); save_rows(arows,AMC); feat_save('E1',FE1)
    e1=pd.DataFrame(rows); am=pd.DataFrame(arows)
    print(f'E1 rows {len(e1)} | unequal-count pairs {(am[am.cost=="A"].nA!=am[am.cost=="A"].nB).mean():.0%} | faithful==P maps {((am[am.cost=="F"].overlap_P==1).mean()):.0%}')
    print(e1.groupby(['cost','layout'])[['real_inc','real_dino','place','lpips_rA','lpips_rB']].median().round(3))
    print(am.groupby('cost')[['A_cost','G_cost','overlap_A','overlap_P']].median().round(3))


In [ ]:
# ============================================================
# 6. E1 SENSITIVITY (§3): 24-pair subset — (a) λ sweep at t=0.5 on all three layouts, full curve reported; (b) t-path for the four main costs
# ============================================================
CSV1S=f'{CORR}/E1_sens.csv'; rows,have=load_rows(CSV1S,['k','cell','t']); FES=feat_load('E1_sens'); t0=time.time()
if RUN['SENS']:
    for n_,k in enumerate(SENS):
        d,EA,EB=pair_elems(k); anc=anchors(k); rA,rB=anc['rA'],anc['rB']
        jobs=[]                                              # (cell, t, lam_or_cost, layout)
        for lam in LAMS_SENS:
            for l in ['morph','fixA','fixB']: jobs.append((f'L{lam}_{l}',0.5,lam,l))
        for c in ['P','A','PA','F']:
            for l in ['morph','fixA','fixB']:
                for t in TS: jobs.append((f'{c}_{l}',t,c,l))
        if all((k,cell,t) in have for cell,t,_,_ in jobs): continue
        cache={}
        def get_asg(dr,key):
            if (dr,key) not in cache:
                ES,ET=(EA,EB) if dr=='a2b' else (EB,EA); cache[(dr,key)]=solve_cost(ES,ET,key)[0] if isinstance(key,str) else solve_cost(ES,ET,'x',lam=key)[0]
            return cache[(dr,key)]
        for cell,t,key,l in jobs:
            fp=img_path('E1_sens',cell,k,t)
            if not os.path.exists(fp):
                if l=='morph': out=decode_moving(d,EA,EB,get_asg('a2b',key),t)
                elif l=='fixA': out=decode_fixed(d,'a',EA,EB,get_asg('a2b',key),t)
                else: out=decode_fixed(d,'b',EB,EA,get_asg('b2a',key),t)
                to_pil(out).save(fp)
            if (k,cell,t) not in have:
                pil=Image.open(fp).convert('RGB'); s,fi,fd=score_image(k,pil); FES[f'{cell}|{k}|{t}|inc']=fi; FES[f'{cell}|{k}|{t}|dino']=fd
                rows.append(dict(k=k,cell=cell,t=t,lam=(key if not isinstance(key,str) else LAM.get(key,float('nan'))),cost=(key if isinstance(key,str) else f'L{key}'),layout=l,regime=pair_regime(k),**s,lpips_rA=lpips_d(pil,rA),lpips_rB=lpips_d(pil,rB)))
        save_rows(rows,CSV1S); feat_save('E1_sens',FES); print(f'{n_+1}/{len(SENS)} {time.time()-t0:.0f}s')
    es=pd.DataFrame(rows)
    print('λ-sweep medians at t=0.5 (real_dino):'); print(es[es.t==0.5].pivot_table(index='lam',columns='layout',values='real_dino',aggfunc='median').round(3))
    print('t-path medians of place (dA/(dA+dB)) for A vs P, morph:'); print(es[es.cost.isin(['A','P','PA'])&(es.layout=='morph')].pivot_table(index='cost',columns='t',values='place',aggfunc='median').round(3))


In [ ]:
# ============================================================
# 7. E2a LATENT COUNTERFACTUALS (§4): identical target feature multiset, controlled spatial reliability; ancestry oracle; decoded oracle error
#    conditions: 40 bases × a∈A_LEVELS × σ∈SIG_LEVELS (d_NN units) × m∈M_FRACS × cost∈{P,A,PA} × layout∈{fixA,morph}
# ============================================================
CSV2=f'{CORR}/E2a.csv'; rows,have=load_rows(CSV2,['k','a','sig','m','cost','layout']); ECSV=f'{CORR}/E2a_endpoints.csv'; erows,ehave=load_rows(ECSV,['k','a','sig','m']); t0=time.time()
if RUN['E2a']:
    for n_,k in enumerate(E2_BASES):
        conds=[(a,s,m) for a in A_LEVELS for s in SIG_LEVELS for m in M_FRACS]
        if all((k,a,s,m,c,l) in have for a,s,m in conds for c in ['P','A','PA'] for l in ['fixA','morph']): continue
        d,EA,_=pair_elems(k); base=to_pil(decode_blob(d['blob_a'])); fA=dino1(base); ref_m0={}
        for a,s,m in conds:
            ET=e2a_target(EA,a,s,m,seed=k); tgt=to_pil(decode_blob(blob_of(d,ET))); fB=dino1(tgt)
            if m==0: ref_m0[(a,s)]=tgt
            if (k,a,s,m) not in ehave:
                sB,_,_=score_generic(tgt); erows.append(dict(k=k,a=a,sig=s,m=m,n=ET['n'],n_moved=ET['n_moved'],disp=ET['disp'],dnn=ET['dnn'],dAB=float(np.linalg.norm(fA-fB)),
                                                             lpips_vs_m0=lpips_d(tgt,ref_m0.get((a,s),tgt)),real_dino_B=sB['real_dino'],real_inc_B=sB['real_inc']))
            Cp,Ca=cost_mats(EA,ET); ora=identity_asg(ET['n']); orc=dict(app=float(Ca[ora['r'],ora['c']].mean()/SC['s_app']),geo=float(Cp[ora['r'],ora['c']].mean()/SC['s_pos']))
            O={'fixA':to_pil(decode_fixed(d,'a',EA,ET,ora,0.5)),'morph':to_pil(decode_moving(d,EA,ET,ora,0.5))}; fO={l:dino1(O[l]) for l in O}
            if k in SAVE_BASES:
                tgt.save(img_path('E2a',f'target_a{a}_s{s}_m{m}',k)); [O[l].save(img_path('E2a',f'oracle_{l}_a{a}_s{s}_m{m}',k)) for l in O]
            for c in ['P','A','PA']:
                asg=solve_cost(EA,ET,c,Cp=Cp,Ca=Ca)[0]; rec=float((asg['map_full']==ora['map_full']).float().mean())
                gap=dict(app_gap=float(Ca[asg['r'],asg['c']].mean()/SC['s_app'])-orc['app'],geo_gap=float(Cp[asg['r'],asg['c']].mean()/SC['s_pos'])-orc['geo'])
                for l in ['fixA','morph']:
                    if (k,a,s,m,c,l) in have: continue
                    pil=to_pil(decode_fixed(d,'a',EA,ET,asg,0.5) if l=='fixA' else decode_moving(d,EA,ET,asg,0.5))
                    if k in SAVE_BASES: pil.save(img_path('E2a',f'{c}_{l}_a{a}_s{s}_m{m}',k))
                    sc,fi,fd=score_generic(pil,fA,fB)
                    rows.append(dict(k=k,a=a,sig=s,m=m,cost=c,layout=l,recovery=rec,**gap,oracle_lpips=lpips_d(pil,O[l]),oracle_dino=float(np.linalg.norm(fd-fO[l])),**sc))
        save_rows(rows,CSV2); save_rows(erows,ECSV); print(f'{n_+1}/{len(E2_BASES)} {time.time()-t0:.0f}s')
    e2=pd.DataFrame(rows); ep=pd.DataFrame(erows)
    print('endpoint validity (median over bases): LPIPS(target_m, target_m0), realism'); print(ep.groupby(['a','sig','m'])[['lpips_vs_m0','real_dino_B','disp']].median().round(3))
    print('ancestry recovery (median):'); print(e2[e2.layout=='fixA'].pivot_table(index=['sig','m'],columns='cost',values='recovery',aggfunc='median').round(2))
    print('oracle LPIPS (median):'); print(e2.pivot_table(index=['layout','sig','m'],columns='cost',values='oracle_lpips',aggfunc='median').round(3))


In [ ]:
# ============================================================
# 8. E2b REAL-IMAGE REPLICATION (§4): same 40 bases; fixed hue/contrast edit × geometry ∈ {id, crop-shift 32px, 2×2 tile derangement, 4×4};
#    both endpoints re-encoded; oracle registration control for the shift (positions corrected in the cost only, decoded with real coordinates)
# ============================================================
HUES=proto['e2b']['hue_deg']; SHIFT=proto['e2b']['shift_px']; TILES=proto['e2b']['tiles']; CONTR=proto['e2b']['contrast']; PAD=16
def prep_full(path,s=IMG_SIZE+2*PAD):
    im=Image.open(path).convert('RGB'); w,h=im.size; r=s/min(w,h); im=im.resize((max(s,round(w*r)),max(s,round(h*r))),Image.BICUBIC); w,h=im.size; l,tp=(w-s)//2,(h-s)//2
    return im.crop((l,tp,l+s,tp+s))
def crop(im,ox,oy): return im.crop((ox,oy,ox+IMG_SIZE,oy+IMG_SIZE))
def hue_shift(im,deg):
    h,s,v=im.convert('HSV').split(); h=h.point(lambda p:(p+int(round(deg/360*255)))%256); return ImageEnhance.Contrast(Image.merge('HSV',(h,s,v)).convert('RGB')).enhance(CONTR)
def tile_derange(im,g,seed):
    n=IMG_SIZE//g; tiles=[im.crop((j*n,i*n,(j+1)*n,(i+1)*n)) for i in range(g) for j in range(g)]; order=np.random.RandomState(seed).permutation(g*g).tolist()
    src={order[q]:order[(q+1)%len(order)] for q in range(len(order))}; out=Image.new('RGB',(IMG_SIZE,IMG_SIZE))
    for q in range(g*g): out.paste(tiles[src[q]],((q%g)*n,(q//g)*n))
    return out
GEOMS=['id',f'shift{SHIFT}']+[f'tile{g}' for g in TILES]
CSV2B=f'{CORR}/E2b.csv'; rows,have=load_rows(CSV2B,['k','hue','geom','cost','layout']); t0=time.time()
if RUN['E2b']:
    for n_,k in enumerate(E2_BASES):
        if all((k,h,gm,c,l) in have for h in HUES for gm in GEOMS for c in ['P','A','PA','Preg'] for l in ['fixA','morph'] if not (c=='Preg' and not gm.startswith('shift'))): continue
        full=prep_full(DTD_BY_NAME[nameA(k)]); A_img=crop(full,PAD,PAD); dA=build_d(cnt_load(A_img),cnt_load(A_img)); EA=elements(dA['blob_a']); rA=to_pil(decode_blob(dA['blob_a'])); fA=dino1(rA)
        for h in HUES:
            edited=hue_shift(full,h)
            for gm in GEOMS:
                if gm=='id': B_img=crop(edited,PAD,PAD); shift=0
                elif gm.startswith('shift'): B_img=crop(edited,PAD+SHIFT,PAD+SHIFT); shift=SHIFT      # content at A-coord X sits at B-coord X-shift
                else: g=int(gm[4:]); B_img=tile_derange(crop(edited,PAD,PAD),g,seed=k*10+g); shift=0
                dB=build_d(cnt_load(B_img),cnt_load(B_img)); EB=elements(dB['blob_a']); d=dict(blob_a=dA['blob_a'],bf_a=dA['bf_a'],si_a=dA['si_a'],blob_b=dB['blob_a'],bf_b=dB['bf_a'],si_b=dB['si_a'])
                rB=to_pil(decode_blob(dB['blob_a'])); fB=dino1(rB); Cp,Ca=cost_mats(EA,EB)
                costs={c:solve_cost(EA,EB,c,Cp=Cp,Ca=Ca) for c in ['P','A','PA']}
                if shift:                                                                       # oracle registration control (cost only)
                    EBreg=dict(EB); EBreg['pos']=EB['pos']+shift/IMG_SIZE; Cpr,_=cost_mats(EA,EBreg); costs['Preg']=solve_cost(EA,EB,'P',Cp=Cpr,Ca=Ca)
                    reg_gain=float(Cp[costs['P'][0]['r'],costs['P'][0]['c']].mean()-Cpr[costs['Preg'][0]['r'],costs['Preg'][0]['c']].mean())/SC['s_pos']
                else: reg_gain=float('nan')
                for c,(asg,_,_) in costs.items():
                    ovA=float((asg['map_full']==costs['A'][0]['map_full']).float().mean())
                    for l in ['fixA','morph']:
                        if (k,h,gm,c,l) in have: continue
                        pil=to_pil(decode_fixed(d,'a',EA,EB,asg,0.5) if l=='fixA' else decode_moving(d,EA,EB,asg,0.5))
                        if k in SAVE_BASES: pil.save(img_path('E2b',f'{c}_{l}_h{h}_{gm}',k)); rB.save(img_path('E2b',f'target_h{h}_{gm}',k))
                        sc,fi,fd=score_generic(pil,fA,fB)
                        rows.append(dict(k=k,hue=h,geom=gm,cost=c,layout=l,nA=EA['n'],nB=EB['n'],overlap_A=ovA,reg_gain=reg_gain,dAB=float(np.linalg.norm(fA-fB)),
                                         A_cost=float(Ca[asg['r'],asg['c']].mean()/SC['s_app']),G_cost=float(Cp[asg['r'],asg['c']].mean()/SC['s_pos']),**sc,lpips_rA=lpips_d(pil,rA),lpips_rB=lpips_d(pil,rB)))
        save_rows(rows,CSV2B); print(f'{n_+1}/{len(E2_BASES)} {time.time()-t0:.0f}s')
    eb=pd.DataFrame(rows)
    print('E2b realism (median) by geometry × cost:'); print(eb.pivot_table(index=['layout','geom'],columns='cost',values='real_dino',aggfunc='median').round(3))
    print('assignment overlap with A-only:'); print(eb[eb.layout=='fixA'].pivot_table(index='geom',columns='cost',values='overlap_A',aggfunc='median').round(2))
    print('registration control: median normalised geometric-cost gain on shift pairs',round(float(eb[eb.geom.str.startswith('shift')].reg_gain.median()),4))


In [ ]:
# ============================================================
# 9. E3 CORRUPT / RESTORE THE MATCHING CUE (§5): position labels corrupted ONLY in the cost (fractions E3_FRACS); appearance descriptors as the
#    matched negative control; restoration must reproduce the originals bit-for-bit; B-relabel sanity. Natural SENS pairs + E2a aligned-ambiguous pairs.
# ============================================================
CSV3=f'{CORR}/E3.csv'; rows,have=load_rows(CSV3,['k','family','kind','frac','cost','layout']); t0=time.time()
def corrupt(E,frac,seed,what):
    """permute rows of the cost-side descriptor ('pos' or 'f') for a seeded nested selection (cyclic derangement). Real parameters untouched."""
    n=E['n']; g=torch.Generator(device=device).manual_seed(int(seed)+101); order=torch.randperm(n,generator=g,device=device); perm=torch.arange(n,device=device)
    nm=max(2,int(math.ceil(frac*n)));
    if nm>=2: S=order[:nm]; perm[S]=S.roll(-1)
    Ec=dict(E); Ec[what]=E[what][perm]; return Ec,int((perm!=torch.arange(n,device=device)).sum())
if RUN['E3']:
    fams=[('natural',k,None) for k in SENS]+[('e2a_aligned',k,(A_LEVELS[0],SIG_LEVELS[1],0.0)) for k in E2_BASES[:24]]
    for n_,(fam,k,cond) in enumerate(fams):
        keys=[(k,fam,kind,fr,c,l) for kind in ['pos','app'] for fr in E3_FRACS for c in ['P','A','PA'] for l in ['fixA','morph']]
        if all(x in have for x in keys): continue
        if fam=='natural': d,EA,EB=pair_elems(k); anc=anchors(k); fA,fB=anc['fA'],anc['fB']
        else:
            d,EA,_=pair_elems(k); EB=e2a_target(EA,*cond,seed=k); fA=dino1(to_pil(decode_blob(d['blob_a']))); fB=dino1(to_pil(decode_blob(blob_of(d,EB))))
        Cp,Ca=cost_mats(EA,EB); base={c:solve_cost(EA,EB,c,Cp=Cp,Ca=Ca)[0] for c in ['P','A','PA']}
        bimg={(c,l):(decode_fixed(d,'a',EA,EB,base[c],0.5) if l=='fixA' else decode_moving(d,EA,EB,base[c],0.5)) for c in base for l in ['fixA','morph']}
        bpil={cl:to_pil(v) for cl,v in bimg.items()}; borig={cl:score_generic(p_)[0]['real_dino'] for cl,p_ in bpil.items()}
        for kind in ['pos','app']:
            for fr in E3_FRACS:
                Ec,nmoved=corrupt(EB,fr,k,'pos' if kind=='pos' else 'f'); Cpc,Cac=cost_mats(EA,Ec)
                if kind=='pos': Cpu,Cau=Cpc,Ca
                else: Cpu,Cau=Cp,Cac
                for c in ['P','A','PA']:
                    asg=solve_cost(EA,EB,c,Cp=Cpu,Ca=Cau)[0]; edges=float((asg['map_full']!=base[c]['map_full']).float().mean())
                    # attributes of reassigned elements: appearance cost of the new edges under the TRUE descriptors
                    tru=dict(A_cost_true=float(Ca[asg['r'],asg['c']].mean()/SC['s_app']),G_cost_true=float(Cp[asg['r'],asg['c']].mean()/SC['s_pos']))
                    for l in ['fixA','morph']:
                        if (k,fam,kind,fr,c,l) in have: continue
                        out=decode_fixed(d,'a',EA,EB,asg,0.5) if l=='fixA' else decode_moving(d,EA,EB,asg,0.5); pil=to_pil(out); sc,fi,fd=score_generic(pil,fA,fB)
                        rows.append(dict(k=k,family=fam,kind=kind,frac=fr,n_corrupted=nmoved,cost=c,layout=l,edges_changed=edges,img_maxdiff=maxdiff(out,bimg[(c,l)]),lpips_vs_orig=lpips_d(pil,bpil[(c,l)]),**tru,**sc,
                                         real_dino_orig=borig[(c,l)]))
        # restoration + B-relabel checks (once per pair): recompute from true metadata -> identical; relabel B consistently -> identical
        rest=max(maxdiff(decode_fixed(d,'a',EA,EB,solve_cost(EA,EB,c,Cp=Cp,Ca=Ca)[0],0.5),bimg[(c,'fixA')]) for c in base)
        pb=torch.as_tensor(np.random.RandomState(k).permutation(EB['n']),device=device); EBp={kk:(v[pb] if kk not in ('bg','u') and torch.is_tensor(v) and v.ndim>=1 and v.shape[0]==EB['n'] else v) for kk,v in EB.items()}
        rel=maxdiff(decode_moving(d,EA,EBp,solve_cost(EA,EBp,'PA')[0],0.5),bimg[('PA','morph')])
        rows.append(dict(k=k,family=fam,kind='restore',frac=0.0,n_corrupted=0,cost='all',layout='all',edges_changed=0.0,img_maxdiff=rest,lpips_vs_orig=0.0,relabel_maxdiff=rel))
        if n_%8==0: save_rows(rows,CSV3); print(f'{n_+1}/{len(fams)} {time.time()-t0:.0f}s')
    save_rows(rows,CSV3); e3=pd.DataFrame(rows); r_=e3[e3.kind=='restore']
    print(f"restoration max diff {r_.img_maxdiff.max():.2e} | B-relabel max diff {r_.relabel_maxdiff.max():.2e} (both must be ~0)")
    e3m=e3[e3.kind!='restore']
    print('edges changed (median) — expected: A-only invariant to pos corruption, P-only invariant to app corruption:'); print(e3m.pivot_table(index=['kind','frac'],columns='cost',values='edges_changed',aggfunc='median').round(2))
    print('realism change vs original (median of real_dino - real_dino_orig):'); e3m=e3m.assign(d_real=e3m.real_dino-e3m.real_dino_orig); print(e3m.pivot_table(index=['family','kind','layout'],columns='cost',values='d_real',aggfunc='median').round(3))


In [ ]:
# ============================================================
# 10. E4 BRIDGE TO THE ACTUAL OPERATOR (§6): Gaussian-level entropic appearance matching at fixed source layout on an ε grid (normalised cost units),
#     variance-matched to dense OT; scored beside every existing dense row on the same 120 pairs (canonical 128 px protocol)
# ============================================================
CSV4=f'{CORR}/E4_gauss_sinkhorn.csv'; rows,have=load_rows(CSV4,['k','eps']); FE4=feat_load('E4'); t0=time.time()
def gauss_sinkhorn_feats(EA,EB,Ca,eps):
    """balanced Sinkhorn with uniform marginals on the active elements (normalised appearance cost); barycentric projection of B's [ptf|rot]"""
    Cn=(Ca/SC['s_app']).double(); n,m=Cn.shape; la,lb=-math.log(n),-math.log(m); fdu=torch.zeros(n,device=device,dtype=torch.float64); gdu=torch.zeros(m,device=device,dtype=torch.float64)
    for it in range(1,3001):
        fdu=eps*(la-torch.logsumexp((gdu[None,:]-Cn)/eps,1)); gdu=eps*(lb-torch.logsumexp((fdu[:,None]-Cn)/eps,0))
        if it%10==0:
            P=torch.exp((fdu[:,None]+gdu[None,:]-Cn)/eps)
            if float((P.sum(1)-1/n).abs().max()*n)<1e-6: break
    P=torch.exp((fdu[:,None]+gdu[None,:]-Cn)/eps); Q=P/(P.sum(1,keepdim=True)+1e-300); fB=torch.cat([EB['ptf'],EB['rot']],-1).double()
    Fstar=(Q@fB).float(); vr=float(Fstar.double().var(0).sum()/fB.var(0).sum()); return Fstar,dict(var_ret=vr,iters=it,**plan_stats(P))
if RUN['E4']:
    for n_,k in enumerate(KS120):
        if all((k,e) in have for e in [str(e_) for e_ in EPS_GRID]+['dense_ot','dense_exact']): continue
        d,EA,EB=pair_elems(k); Cp,Ca=cost_mats(EA,EB)
        if (k,'dense_ot') not in have:
            fo=field(d,'ot',k); fe=field(d,'exact',k)
            rows.append(dict(k=k,eps='dense_ot',var_ret=var_ret(fo['U'],fo['Y']),**{f'dense_{kk}':v for kk,v in fo['info'].items() if isinstance(v,(int,float))})); rows.append(dict(k=k,eps='dense_exact',var_ret=var_ret(fe['U'],fe['Y'])))
        for e in EPS_GRID:
            fp=img_path('E4',f'gA_sink{e}_fixA',k)
            if (k,str(e)) in have and os.path.exists(fp): continue
            Fs,info=gauss_sinkhorn_feats(EA,EB,Ca,e); pil=to_pil(decode_fixed_feats(d,'a',EA,Fs,EB['bg'],0.5)); pil.save(fp); s,fi,fd=score_image(k,pil)
            FE4[f'gA_sink{e}_fixA|{k}|inc']=fi; FE4[f'gA_sink{e}_fixA|{k}|dino']=fd; rows.append(dict(k=k,eps=str(e),**info,**s))
        if n_%20==0: save_rows(rows,CSV4); feat_save('E4',FE4); print(f'{n_+1}/{len(KS120)} {time.time()-t0:.0f}s')
    save_rows(rows,CSV4); feat_save('E4',FE4); e4=pd.DataFrame(rows)
    e4['eps']=e4.eps.astype(str); vr=e4.groupby('eps').var_ret.median(); dense_vr=float(vr['dense_ot']); EPS_MATCH=min(EPS_GRID,key=lambda e:abs(float(vr[str(e)])-dense_vr))
    print('median retained variance per ε:',{str(e):round(float(v),3) for e,v in vr.items()},f'-> ε matched to dense OT: {EPS_MATCH}')
    json.dump(dict(eps_match=EPS_MATCH,var_ret=vr.astype(float).to_dict()),open(f'{CORR}/E4_match.json','w'),indent=1)
    # --- score existing dense rows + controls on the 120 (fresh, canonical 128 px) ---
    DENSE={m:p for m,p in {'ot':f'{SCALE}/ot','exact':f'{SCALE}/exact','ot_sym':f'{SCALE}/ot_sym','pixel_linear':f'{SCALE}/pixel_linear','texton_linear':f'{SCALE}/texton_linear','gaussian_w2':f'{SCALE}/gaussian_w2',
                            'target_mean':f'{SCALE}/target_mean','source_copy':f'{SCALE}/source_copy','target_copy':f'{SCALE}/target_copy','tm':f'{SCALE}/mid','gpt':f'{SCALE}/gpt'}.items() if os.path.isdir(p)}
    FED=feat_load('dense'); drows=[]
    for m,pdir in DENSE.items():
        for k in KS120:
            fp=f'{pdir}/{k:04d}.png'
            if not os.path.exists(fp): continue
            if f'{m}|{k}|inc' not in FED:
                pil=Image.open(fp).convert('RGB'); s,fi,fd=score_image(k,pil); FED[f'{m}|{k}|inc']=fi; FED[f'{m}|{k}|dino']=fd; FED[f'{m}|{k}|meta']=np.array([s['real_inc'],s['real_dino'],s['dA'],s['dB'],s['place']])
            s_=FED[f'{m}|{k}|meta']; drows.append(dict(k=k,cell=m,real_inc=s_[0],real_dino=s_[1],dA=s_[2],dB=s_[3],place=s_[4]))
    feat_save('dense',FED); pd.DataFrame(drows).to_csv(f'{CORR}/dense_rows.csv',index=False); print('dense rows scored:',list(DENSE))


In [ ]:
# ============================================================
# 11. ANALYSIS (§7, §8): paired contrasts, Holm, cost×layout (H4) and reliability (H2/H3) interactions, cluster bootstrap over endpoint-reuse
#     components / bases, paired m-of-n ΔKID vs the full reference, endpoint retrieval with controls -> summary.json
# ============================================================
from scipy.stats import spearmanr
S={'protocol':proto['version'],'frozen_at':proto['frozen_at']}
def cboot(x,cl,n=4000,seed=0):
    """cluster bootstrap of the mean: resample clusters with replacement (x aligned with cluster ids cl)"""
    x=np.asarray(x,float); cl=np.asarray(cl); ids=np.unique(cl); groups=[x[cl==i] for i in ids]; rng=np.random.RandomState(seed); vals=[]
    for _ in range(n):
        pick=rng.randint(len(groups),size=len(groups)); vals.append(np.concatenate([groups[i] for i in pick]).mean())
    v=np.array(vals); return float(x.mean()),float(np.percentile(v,2.5)),float(np.percentile(v,97.5)),float(2*min((v<=0).mean(),(v>=0).mean()))
def holm(ps):
    ps=np.asarray(ps,float); o=np.argsort(ps); adj=np.empty_like(ps); m=len(ps); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*ps[i]); adj[i]=min(1,run)
    return adj
def kid_ub(X,Y,kyy_sum=None):
    dd=X.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3; m=len(X); n=len(Y); kxx=kk(X,X); kxy=kk(X,Y); kyy_s=kyy_sum if kyy_sum is not None else ((kk(Y,Y).sum()-n)/(n*(n-1)))
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+kyy_s-2*kxy.mean())
REF=ref_inc.astype(np.float64); _kyy=((REF@REF.T/REF.shape[1]+1)**3); KYY=(_kyy.sum()-len(REF))/(len(REF)*(len(REF)-1)); del _kyy
def fid(X,Y):
    mx,my=X.mean(0),Y.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(Y,rowvar=False); cm,_=linalg.sqrtm(Sx@Sy,disp=False); cm=cm.real if np.iscomplexobj(cm) else cm
    return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*cm))
def stackf(FE,cell,ks,kind,tsuf=''): return np.stack([FE[f'{cell}|{k}{tsuf}|{kind}'] for k in ks]).astype(np.float64)
def kid_paired(FE,cells,ks,m=100,B=200,seed=0,tsuf=''):
    """point KID on all ks + paired m-of-n subsampling with shared draws -> per-cell draws (for ΔKID CIs)"""
    X={c:stackf(FE,c,ks,'inc',tsuf) for c in cells}; rng=np.random.RandomState(seed); draws={c:[] for c in cells}
    for _ in range(B):
        idx=rng.choice(len(ks),m,replace=False)
        for c in cells: draws[c].append(kid_ub(X[c][idx],REF,KYY))
    return {c:kid_ub(X[c],REF,KYY) for c in cells},{c:np.array(v) for c,v in draws.items()},{c:fid(X[c],REF) for c in cells}
def dkid(draws,pt,a,b):
    dv=draws[a]-draws[b]; return dict(dKID=pt[a]-pt[b],lo=float(np.percentile(dv,2.5)),hi=float(np.percentile(dv,97.5)),p_boot=float(2*min((dv<=0).mean(),(dv>=0).mean())))
def retrieval(fd_by_k,side,ks,hard_min=5):
    """rank of the true endpoint (recon) among all other pairs' endpoints (easy) and among same-DTD-category endpoints (hard) by DINO distance"""
    Fq=np.stack([fd_by_k[k] for k in ks]); T=np.stack([anchors(k)['f'+side] for k in ks]); cats=np.array([CAT[k][side] for k in ks]); D=np.linalg.norm(Fq[:,None]-T[None],axis=-1)
    true=D[np.arange(len(ks)),np.arange(len(ks))]; rank=(D<true[:,None]).sum(1)+1; out=dict(mrr=1/rank,top1=(rank==1).astype(float)); hard=[]
    for i,k in enumerate(ks):
        same=np.nonzero(cats==cats[i])[0]
        hard.append(np.nan if len(same)<hard_min else 1/((D[i,same]<true[i]).sum()+1))
    out['mrr_hard']=np.array(hard); return out

# ---------------- E1 natural pairs ----------------
if os.path.exists(CSV1):
    e1=pd.read_csv(CSV1); ks=sorted(set(e1.k)&KSET); e1=e1[e1.k.isin(ks)]; FE1=feat_load('E1'); cl=np.array([CLUSTER[k] for k in ks])
    e1['bal']=(e1.place-0.5).abs()
    FIX=e1[e1.layout.isin(['fixA','fixB'])].groupby(['k','cost'])[['real_inc','real_dino','bal','lpips_rA','lpips_rB','place']].mean().reset_index().assign(layout='fix')
    E=pd.concat([e1[e1.layout=='morph'][FIX.columns],FIX]); OUT1={}; prim=[]
    for l in ['morph','fix']:
        pv={o:E[E.layout==l].pivot(index='k',columns='cost',values=o).loc[ks] for o in ['real_dino','real_inc','bal','lpips_rA','lpips_rB']}
        for o,tab in pv.items():
            for a,b in [('A','PA'),('A','P'),('PA','P')]:
                x=(tab[a]-tab[b]).values; mu,lo,hi,pb=cboot(x,cl); pw=float(wilcoxon(x).pvalue) if np.abs(x).sum()>0 else 1.0
                OUT1[f'{l}|{o}|{a}-{b}']=dict(mean=mu,lo=lo,hi=hi,p_boot=pb,p_wilcoxon=pw,n=len(x))
        # faithful vs P/morph
        if l=='morph':
            x=(pv['real_dino']['F']-pv['real_dino']['P']).values; OUT1['morph|real_dino|F-P']=dict(**dict(zip(['mean','lo','hi','p_boot'],cboot(x,cl))),n=len(x))
    fam=[f'{l}|real_dino|{c}' for l in ['morph','fix'] for c in ['A-PA','A-P']]; adj=holm([OUT1[f]['p_wilcoxon'] for f in fam])
    for f,pa in zip(fam,adj): OUT1[f]['p_holm']=float(pa)
    # H4: cost × layout interaction per pair
    H4={}
    for o in ['real_dino','bal']:
        pm=E[E.layout=='morph'].pivot(index='k',columns='cost',values=o).loc[ks]; pf=E[E.layout=='fix'].pivot(index='k',columns='cost',values=o).loc[ks]
        for b in ['PA','P']:
            x=((pf['A']-pf[b])-(pm['A']-pm[b])).values; H4[f'{o}|(A-{b})fix-(A-{b})morph']=dict(zip(['mean','lo','hi','p_boot'],cboot(x,cl)))
    # KID / FID (Inception, vs full reference) per cell, paired ΔKID vs A within layout
    cells=[f'{c}_{l}' for c in ['P','A','PA'] for l in ['morph','fixA','fixB']]+['F_morph']; ptk,drk,fidk=kid_paired(FE1,cells,ks)
    KID={c:dict(KID=ptk[c],FID=fidk[c]) for c in cells}
    for l in ['morph','fixA','fixB']:
        for b in ['PA','P']: KID[f'A_{l}-{b}_{l}']=dkid(drk,ptk,f'A_{l}',f'{b}_{l}')
    KID['A_morph-F_morph']=dkid(drk,ptk,'A_morph','F_morph')
    # retrieval with controls
    RET={}
    for c in cells:
        fdk={k:FE1[f'{c}|{k}|dino'] for k in ks}; rB=retrieval(fdk,'B',ks); rA=retrieval(fdk,'A',ks); RET[c]=dict(mrr_B=float(np.mean(rB['mrr'])),top1_B=float(np.mean(rB['top1'])),mrr_hard_B=float(np.nanmean(rB['mrr_hard'])),mrr_A=float(np.mean(rA['mrr'])))
        RET[c]['_mrrB']=rB['mrr']
    for l in ['morph','fixA','fixB']:
        for b in ['PA','P']: x=RET[f'A_{l}']['_mrrB']-RET[f'{b}_{l}']['_mrrB']; RET[f'A_{l}-{b}_{l}|mrr_B']=dict(zip(['mean','lo','hi','p_boot'],cboot(x,cl)))
    if os.path.exists(f'{FEATS}/dense.npz'):
        FED=feat_load('dense')
        for m in ['source_copy','target_copy','target_mean','ot','exact','texton_linear']:
            if all(f'{m}|{k}|dino' in FED for k in ks): r_=retrieval({k:FED[f'{m}|{k}|dino'] for k in ks},'B',ks); RET[f'ctrl_{m}']=dict(mrr_B=float(np.mean(r_['mrr'])),top1_B=float(np.mean(r_['top1'])),mrr_hard_B=float(np.nanmean(r_['mrr_hard'])))
    for c in cells: RET[c].pop('_mrrB')
    # regime strata (secondary)
    STR={}
    for rg in ['both_periodic','both_stochastic','mixed']:
        kk=[k for k in ks if pair_regime(k)==rg]
        if len(kk)>=8:
            for l in ['morph','fix']:
                tab=E[(E.layout==l)&E.k.isin(kk)].pivot(index='k',columns='cost',values='real_dino'); STR[f'{rg}|{l}|A-PA']=dict(n=len(kk),**dict(zip(['mean','lo','hi','p_boot'],cboot((tab['A']-tab['PA']).values,[CLUSTER[k] for k in tab.index]))))
    S['E1']=dict(contrasts=OUT1,H4=H4,KID=KID,retrieval=RET,regime_strata=STR,n_pairs=len(ks))
    print('=== E1 natural pairs: A minus PA / P (positive = appearance-only better), DINO realism, cluster-bootstrap 95% CI, Holm-adjusted Wilcoxon ===')
    for f in fam: r_=OUT1[f]; print(f"{f:28} {r_['mean']:+.4f} [{r_['lo']:+.4f},{r_['hi']:+.4f}]  p_holm={r_['p_holm']:.3f}")
    print('H4 cost×layout:',{k_:f"{v['mean']:+.4f} [{v['lo']:+.4f},{v['hi']:+.4f}]" for k_,v in H4.items()})
    print('KID:',{c:round(v['KID'],4) for c,v in KID.items() if 'KID' in v}); print('ΔKID (A−X<0 means A better):',{c:f"{v['dKID']:+.4f} [{v['lo']:+.4f},{v['hi']:+.4f}]" for c,v in KID.items() if 'dKID' in v})
    print('retrieval MRR_B:',{c:round(v['mrr_B'],3) for c,v in RET.items() if 'mrr_B' in v})
# ---------------- E1 sensitivity: λ curve and t-path ----------------
if os.path.exists(CSV1S):
    es=pd.read_csv(CSV1S); es['bal']=(es.place-0.5).abs(); lamt=es[es.cost.str.startswith('L')&(es.t==0.5)]
    LAMC={o:lamt.pivot_table(index='lam',columns='layout',values=o,aggfunc='median').round(4).to_dict() for o in ['real_dino','real_inc','bal','lpips_rA','lpips_rB']}
    tp=es[es.cost.isin(['P','A','PA','F'])]; TPATH={}
    for (c,l),g_ in tp.groupby(['cost','layout']):
        pv=g_.pivot(index='k',columns='t',values='place'); rho=[spearmanr(pv.columns.values,pv.loc[k].values).correlation for k in pv.index]
        TPATH[f'{c}_{l}']=dict(place_by_t=pv.median().round(3).to_dict(),place_monotone_rho_median=float(np.nanmedian(rho)),dB_at_t1=float(g_[g_.t==1.0].dB.median()),lpips_rB_at_t1=float(g_[g_.t==1.0].lpips_rB.median()))
    S['E1_sens']=dict(lambda_curves=LAMC,t_path=TPATH); print('λ curve (real_dino, median):'); print(pd.DataFrame(LAMC['real_dino']))
# ---------------- E2a: reliability interaction (H2), ambiguity moderation (H3) ----------------
if os.path.exists(CSV2):
    e2=pd.read_csv(CSV2); e2['Q']=-e2.oracle_lpips; H2={}; CURVES={}; REC={}
    for l in ['fixA','morph']:
        for a in A_LEVELS:
            for s in SIG_LEVELS:
                sub=e2[(e2.layout==l)&(e2.a==a)&(e2.sig==s)]; pv=sub.pivot_table(index=['k','m'],columns='cost',values='Q'); dg=(pv['PA']-pv['A']).unstack('m'); dp=(pv['P']-pv['A']).unstack('m')
                CURVES[f'{l}|a{a}|s{s}']=dict(dgeo_PA_mean=dg.mean().round(4).to_dict(),dgeo_PA_lo=dg.quantile(0.025).round(4).to_dict(),dgeo_PA_hi=dg.quantile(0.975).round(4).to_dict(),dgeo_P_mean=dp.mean().round(4).to_dict(),
                                              recovery={c:sub.pivot_table(index='m',columns='cost',values='recovery',aggfunc='mean')[c].round(3).to_dict() for c in ['P','A','PA']})
                x=(dg[0.0]-dg[1.0]).dropna().values; H2[f'{l}|a{a}|s{s}|PA']=dict(**dict(zip(['mean','lo','hi','p_boot'],cboot(x,np.arange(len(x))))),p_wilcoxon=float(wilcoxon(x).pvalue) if np.abs(x).sum()>0 else 1.0,n_bases=len(x))
                x=(dp[0.0]-dp[1.0]).dropna().values; H2[f'{l}|a{a}|s{s}|P']=dict(**dict(zip(['mean','lo','hi','p_boot'],cboot(x,np.arange(len(x))))),n_bases=len(x))
    H3={}
    for l in ['fixA','morph']:
        for a in A_LEVELS:
            pv=e2[(e2.layout==l)&(e2.a==a)].pivot_table(index=['k','sig','m'],columns='cost',values='Q'); dg=(pv['PA']-pv['A']).unstack(['sig','m'])
            x=((dg[(SIG_LEVELS[1],0.0)]-dg[(SIG_LEVELS[1],1.0)])-(dg[(SIG_LEVELS[0],0.0)]-dg[(SIG_LEVELS[0],1.0)])).dropna().values
            H3[f'{l}|a{a}']=dict(**dict(zip(['mean','lo','hi','p_boot'],cboot(x,np.arange(len(x))))),n_bases=len(x))
    sanity=dict(A_app_gap_max=float(e2[e2.cost=='A'].app_gap.max()),P_geo_gap_max=float(e2[e2.cost=='P'].geo_gap.max()))
    S['E2a']=dict(H2_interaction=H2,H3_ambiguity_moderation=H3,curves=CURVES,sanity=sanity,endpoints=pd.read_csv(ECSV).groupby(['a','sig','m'])[['lpips_vs_m0','real_dino_B','disp','n_moved']].median().round(4).reset_index().to_dict('records'))
    print('=== E2a H2: Δgeo(aligned) − Δgeo(fully disrupted), Q=−oracle LPIPS, PA vs A (positive = geometry helps more when reliable) ===')
    for k_,v in H2.items():
        if k_.endswith('|PA'): print(f"{k_:24} {v['mean']:+.4f} [{v['lo']:+.4f},{v['hi']:+.4f}] p={v['p_wilcoxon']:.3f} n={v['n_bases']}")
    print('H3 (ambiguous − clear):',{k_:f"{v['mean']:+.4f} [{v['lo']:+.4f},{v['hi']:+.4f}]" for k_,v in H3.items()}); print('sanity (must be ≤0):',sanity)
# ---------------- E2b ----------------
if os.path.exists(CSV2B):
    eb=pd.read_csv(CSV2B); B2={}; GEOMS=list(eb.geom.unique())
    for l in ['fixA','morph']:
        pv=eb[eb.layout==l].pivot_table(index=['k','hue','geom'],columns='cost',values='real_dino')
        for gm in GEOMS:
            for b in ['PA','P']:
                x=(pv['A']-pv[b]).xs(gm,level='geom').groupby('k').mean().values; B2[f'{l}|{gm}|A-{b}']=dict(zip(['mean','lo','hi','p_boot'],cboot(x,np.arange(len(x)))))
            if gm.startswith('shift') and 'Preg' in pv.columns:
                x=(pv['Preg']-pv['P']).xs(gm,level='geom').groupby('k').mean().values; B2[f'{l}|{gm}|Preg-P']=dict(zip(['mean','lo','hi','p_boot'],cboot(x,np.arange(len(x)))))
    B2['overlap_with_A']=eb[eb.layout=='fixA'].pivot_table(index='geom',columns='cost',values='overlap_A',aggfunc='median').round(3).to_dict(); B2['reg_gain_median']=float(eb[eb.geom.str.startswith('shift')].reg_gain.median())
    S['E2b']=B2; print('E2b A−X realism by geometry:',{k_:f"{v['mean']:+.4f} [{v['lo']:+.4f},{v['hi']:+.4f}]" for k_,v in B2.items() if isinstance(v,dict) and 'mean' in v})
# ---------------- E3 ----------------
if os.path.exists(CSV3):
    e3=pd.read_csv(CSV3); m_=e3[e3.kind!='restore'].copy(); m_['d_real']=m_.real_dino-m_.real_dino_orig
    S['E3']=dict(restore_maxdiff=float(e3[e3.kind=='restore'].img_maxdiff.max()),relabel_maxdiff=float(e3[e3.kind=='restore'].relabel_maxdiff.max()),
                 edges_changed=m_.pivot_table(index=['kind','frac'],columns='cost',values='edges_changed',aggfunc='median').round(3).reset_index().to_dict('records'),
                 d_realism=m_.pivot_table(index=['family','kind','layout'],columns='cost',values='d_real',aggfunc='median').round(4).reset_index().to_dict('records'),
                 lpips_vs_orig=m_.pivot_table(index=['kind','layout'],columns='cost',values='lpips_vs_orig',aggfunc='median').round(4).reset_index().to_dict('records'),
                 rho_edges_vs_dreal={c:float(spearmanr(g_.edges_changed,g_.d_real,nan_policy='omit').correlation) for c,g_ in m_[m_.kind=='pos'].groupby('cost') if g_.edges_changed.std()>0})
    print('E3 restore/relabel max diff:',S['E3']['restore_maxdiff'],S['E3']['relabel_maxdiff'],'| Spearman(edges changed, Δrealism) per cost:',S['E3']['rho_edges_vs_dreal'])
# ---------------- E4 bridge table ----------------
if os.path.exists(CSV4) and os.path.exists(f'{CORR}/dense_rows.csv') and os.path.exists(CSV1):
    e4=pd.read_csv(CSV4); EM=json.load(open(f'{CORR}/E4_match.json'))['eps_match']; FE4=feat_load('E4'); FED=feat_load('dense'); dr_=pd.read_csv(f'{CORR}/dense_rows.csv'); ks=sorted(set(pd.read_csv(CSV1).k)&KSET)
    ROWS={}; FEALL={}
    for c in ['P_fixA','A_fixA','PA_fixA','A_morph','F_morph']: FEALL[c]=(FE1,c)
    for e in EPS_GRID: FEALL[f'gA_sink{e}_fixA']=(FE4,f'gA_sink{e}_fixA')
    for m in sorted({x.split('|')[0] for x in FED}): FEALL[m]=(FED,m)
    ok={c:[k for k in ks if f'{n}|{k}|inc' in FE] for c,(FE,n) in FEALL.items()}; common=[k for k in ks if all(k in ok[c] for c in FEALL)]
    pt,drw,fd_=kid_paired({f'{n}|{k}|{kind}':FE[f'{n}|{k}|{kind}'] for c,(FE,n) in FEALL.items() for k in common for kind in ['inc','dino']},list(FEALL),common)
    meta=pd.concat([pd.read_csv(CSV1).assign(cell=lambda x:x.cell),dr_]) ; meta=meta[meta.k.isin(common)]
    for c,(FE,n) in FEALL.items():
        sub=meta[meta.cell==n]; r_=retrieval({k:FE[f'{n}|{k}|dino'] for k in common},'B',common)
        ROWS[c]=dict(KID=pt[n],FID=fd_[n],real_inc=float(sub.real_inc.median()) if len(sub) else float('nan'),real_dino=float(sub.real_dino.median()) if len(sub) else float('nan'),bal=float((sub.place-0.5).abs().median()) if len(sub) else float('nan'),mrr_B=float(np.mean(r_['mrr'])),mrr_hard_B=float(np.nanmean(r_['mrr_hard'])))
    ROWS[f'gA_sink{EM}_fixA']['matched_to_dense']=True
    for e in EPS_GRID: ROWS[f'gA_sink{e}_fixA']['var_ret']=float(e4[e4.eps.astype(str)==str(e)].var_ret.median())
    if 'ot' in ROWS: ROWS['ot']['var_ret']=float(e4[e4.eps.astype(str)=='dense_ot'].var_ret.median())
    S['E4']=dict(rows=ROWS,eps_match=EM,dKID_gaussA_vs_denseOT=dkid(drw,pt,f'gA_sink{EM}_fixA','ot') if 'ot' in ROWS else None,dKID_gaussA_hung_vs_dense_exact=dkid(drw,pt,'A_fixA','exact') if 'exact' in ROWS else None,n_common=len(common))
    print('=== E4 bridge (n=%d common pairs) ==='%len(common)); print(pd.DataFrame(ROWS).T[['KID','FID','real_inc','real_dino','bal','mrr_B','mrr_hard_B']].round(4).to_string())
json.dump(S,open(f'{CORR}/summary.json','w'),indent=1,default=lambda o:o.tolist() if hasattr(o,'tolist') else str(o)); print('summary.json written')


In [ ]:
# ============================================================
# 12. FIGURES (§11): factorial panel, reliability panel, Gaussian-vs-dense panel, λ curves, uncurated first-ID grids
# ============================================================
S=json.load(open(f'{CORR}/summary.json'))
if 'E1' in S:
    e1=pd.read_csv(CSV1); e1=e1[e1.k.isin(KSET)]; FIXm=e1[e1.layout.isin(['fixA','fixB'])].groupby(['k','cost']).real_dino.mean().reset_index().assign(layout='fix'); E=pd.concat([e1[e1.layout=='morph'][FIXm.columns],FIXm])
    fig,ax=plt.subplots(1,3,figsize=(14,3.8))
    for i,l in enumerate(['morph','fix']):
        for c in ['P','PA','A']+(['F'] if l=='morph' else []):
            x=E[(E.layout==l)&(E.cost==c)].real_dino.values; mu,lo,hi,_=cboot(x,np.arange(len(x)),n=1000); ax[0].errorbar(i+{'P':-.25,'PA':-.08,'A':.08,'F':.25}[c],mu,[[mu-lo],[hi-mu]],fmt='o',capsize=3,label=c if i==0 else None)
    ax[0].set_xticks([0,1]); ax[0].set_xticklabels(['moving layout','frozen layout']); ax[0].set_ylabel('DINO realism (mean, 95% CI)'); ax[0].legend(fontsize=8); ax[0].set_title('E1 factorial')
    K=S['E1']['KID']; labs=[c for c in K if 'dKID' in K[c]]; ax[1].barh(range(len(labs)),[K[c]['dKID'] for c in labs],xerr=[[K[c]['dKID']-K[c]['lo'] for c in labs],[K[c]['hi']-K[c]['dKID'] for c in labs]],capsize=3)
    ax[1].set_yticks(range(len(labs))); ax[1].set_yticklabels(labs,fontsize=7); ax[1].axvline(0,color='k',lw=.8); ax[1].set_xlabel('ΔKID (A − other), <0 favours A'); ax[1].set_title('paired ΔKID')
    R=S['E1']['retrieval']; labs=[c for c in R if 'mrr_B' in R[c]]; ax[2].bar(range(len(labs)),[R[c]['mrr_B'] for c in labs]); ax[2].set_xticks(range(len(labs))); ax[2].set_xticklabels(labs,rotation=75,fontsize=7); ax[2].set_ylabel('target retrieval MRR'); ax[2].set_title('endpoint retrieval (+controls)')
    plt.tight_layout(); plt.savefig(f'{CFIG}/fig_factorial.png',dpi=170); plt.show()
if 'E2a' in S:
    C=S['E2a']['curves']; fig,ax=plt.subplots(1,2,figsize=(10,3.8))
    for i,l in enumerate(['fixA','morph']):
        for a in A_LEVELS:
            for s in SIG_LEVELS:
                v=C[f'{l}|a{a}|s{s}']; ms=sorted(float(m) for m in v['dgeo_PA_mean']); y=[v['dgeo_PA_mean'][str(m) if str(m) in v['dgeo_PA_mean'] else m] for m in ms]
                lo=[v['dgeo_PA_lo'][str(m) if str(m) in v['dgeo_PA_lo'] else m] for m in ms]; hi=[v['dgeo_PA_hi'][str(m) if str(m) in v['dgeo_PA_hi'] else m] for m in ms]
                ax[i].plot(ms,y,marker='o',ls='-' if a==A_LEVELS[0] else '--',label=f'a={a} σ={s}'); ax[i].fill_between(ms,lo,hi,alpha=.12)
        ax[i].axhline(0,color='k',lw=.8); ax[i].set_xlabel('fraction of supports exchanged (spatial unreliability)'); ax[i].set_ylabel('Δgeo = Q(PA) − Q(A), Q = −oracle LPIPS'); ax[i].set_title(f'E2a predictive test — {l}'); ax[i].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f'{CFIG}/fig_reliability.png',dpi=170); plt.show()
if 'E4' in S:
    R=pd.DataFrame(S['E4']['rows']).T; fig,ax=plt.subplots(1,2,figsize=(11,3.8)); R=R.sort_values('KID')
    ax[0].barh(range(len(R)),R.KID.astype(float)); ax[0].set_yticks(range(len(R))); ax[0].set_yticklabels(R.index,fontsize=7); ax[0].set_xlabel('KID vs full DTD reference'); ax[0].set_title('E4: Gaussian-level vs dense rows')
    g=[c for c in R.index if c.startswith('gA_sink')]; ax[1].plot([float(c.split('sink')[1].split('_')[0]) for c in g],[R.loc[c,'var_ret'] for c in g],marker='o',label='Gaussian appearance Sinkhorn')
    if 'ot' in R.index and pd.notna(R.loc['ot','var_ret']): ax[1].axhline(float(R.loc['ot','var_ret']),color='r',ls='--',label='dense OT ε=0.05')
    ax[1].set_xscale('log'); ax[1].set_xlabel('ε (normalised cost units)'); ax[1].set_ylabel('retained target variance'); ax[1].legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{CFIG}/fig_bridge.png',dpi=170); plt.show()
if 'E1_sens' in S:
    L=S['E1_sens']['lambda_curves']; fig,ax=plt.subplots(1,3,figsize=(13,3.5))
    for a_,o in zip(ax,['real_dino','bal','lpips_rB']):
        for l,v in L[o].items(): xs=sorted(float(x) for x in v); a_.plot(xs,[v[str(x) if str(x) in v else x] for x in xs],marker='o',label=l)
        a_.set_xlabel('λ (0 = appearance only, 1 = position only)'); a_.set_ylabel(o); a_.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f'{CFIG}/fig_lambda.png',dpi=170); plt.show()
# uncurated grids: first 8 pair IDs of the 120 (by k), and the first E2a base
show=sorted(KS120)[:8]; cols=[('A',f'{SCALE}/A')]+[(c,f'{IMGS}/E1/{c}') for c in ['P_morph','A_morph','PA_morph','F_morph','P_fixA','A_fixA','PA_fixA']]+[('dense OT',f'{SCALE}/ot'),('B',f'{SCALE}/B')]
fig,ax=plt.subplots(len(show),len(cols),figsize=(1.6*len(cols),1.6*len(show)))
for i,k in enumerate(show):
    for j,(t_,p) in enumerate(cols):
        fp=f'{p}/{k:04d}.png'; ax[i,j].axis('off')
        if os.path.exists(fp): ax[i,j].imshow(Image.open(fp).convert('RGB').resize((160,160)))
        if i==0: ax[i,j].set_title(t_,fontsize=7)
plt.tight_layout(); plt.savefig(f'{CORR}/qualitative_first_ids.png',dpi=160,bbox_inches='tight'); plt.show()
k0=E2_BASES[0]; a0,s0=A_LEVELS[0],SIG_LEVELS[1]; cols_=['target']+[f'{c}_{l}' for l in ['fixA','morph'] for c in ['P','A','PA']]+['oracle_fixA','oracle_morph']
fig,ax=plt.subplots(len(M_FRACS),len(cols_),figsize=(1.5*len(cols_),1.5*len(M_FRACS)))
for i,m in enumerate(M_FRACS):
    for j,c in enumerate(cols_):
        fp=f'{IMGS}/E2a/{c}_a{a0}_s{s0}_m{m}/{k0:04d}.png'; ax[i,j].axis('off')
        if os.path.exists(fp): ax[i,j].imshow(Image.open(fp).convert('RGB').resize((150,150)))
        if i==0: ax[i,j].set_title(c,fontsize=7)
        if j==0: ax[i,j].text(-0.1,0.5,f'm={m}',transform=ax[i,j].transAxes,fontsize=7,rotation=90,va='center')
plt.tight_layout(); plt.savefig(f'{CFIG}/e2a_base{k0}_grid.png',dpi=160,bbox_inches='tight'); plt.show(); print('figures ->',CFIG)
